<a href="https://colab.research.google.com/github/bulut19/stockout-aware-forecasting/blob/main/stream4_foundation_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stream 4: Foundation models (TimesFM-3 and Chronos-2)


## Step 0: Setup
Connect Google Drive (shared data) and pull the team repo (shared loader and metrics).

In [1]:
# connect Drive and pull the repo
from google.colab import drive
import os
drive.mount('/content/drive')

repo = 'stockout-aware-forecasting'
if not os.path.exists(f'/content/{repo}'):
    !git clone https://github.com/bulut19/{repo}.git /content/{repo}
%cd /content/{repo}
!git pull

Mounted at /content/drive
Cloning into '/content/stockout-aware-forecasting'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 114 (delta 43), reused 44 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.62 MiB | 1.46 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/stockout-aware-forecasting
Already up to date.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1: Load the data
- `daily_demand_v1.parquet`: 2,000 products × 97 days, with recorded `sales` and corrected `demand`
- `forecasts_stream2_v1.parquet`: stream 2's forecasts for the valid and test weeks, our comparison point

Every model is scored against real `sales` on in-stock days only (`oos_hours == 0`), since true demand is unknown when the shelf is empty.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.load_data import PROC_DIR
from src.metrics import wape, mae, bias

print("Data folder:", PROC_DIR)

# The same 2,000 products
sample_ids = pd.read_csv(PROC_DIR / 'sample_ids.csv').series_id.tolist()

# One row per product per day, recorded sales, corrected demand, empty-shelf hours, split
dd = pd.read_parquet(PROC_DIR / 'daily_demand_v1.parquet')
dd['dt'] = pd.to_datetime(dd['dt'])

# Stream 2 forecasts - our comparison point
fc2 = pd.read_parquet(PROC_DIR / 'forecasts_stream2_v1.parquet')

print("Products:", len(sample_ids))
print("Daily table:", dd.shape, "| columns:", list(dd.columns))
print(dd.groupby('set').dt.agg(['min', 'max', 'nunique']))
print("\nStream 2 forecasts:", fc2.shape, "| columns:", list(fc2.columns))
print(fc2.groupby(['model', 'version']).size())

Data folder: /content/drive/MyDrive/MIS 382 Deep Learning and ML/Final Project/Stockout Project Data/processed
Products: 2000
Daily table: (194000, 8) | columns: ['series_id', 'dt', 'sales', 'demand', 'lost_demand', 'oos_hours', 'set', 'zero_sales_day']
             min        max  nunique
set                                 
test  2024-06-26 2024-07-02        7
train 2024-03-28 2024-06-11       76
valid 2024-06-12 2024-06-25       14

Stream 2 forecasts: (168000, 14) | columns: ['series_id', 'dt', 'set', 'sales', 'demand', 'oos_hours', 'stockout_band', 'first_category_id', 'version', 'model', 'forecast', 'q10', 'q50', 'q90']
model           version  
lightgbm        corrected    42000
                raw          42000
seasonal_naive  corrected    42000
                raw          42000
dtype: int64


## Step 2: Reproduce stream 2's scores
If we match stream 2's numbers, we know our scoring is identical. Target: LightGBM raw 0.389, corrected 0.350; seasonal naive 0.385.

**Fair comparison rule:** stream 2 forecasts every day exactly 7 days ahead (to predict June 30, it only sees data up to June 23). Our foundation models must follow the same rule, or they'd win just by seeing more recent sales.

In [4]:
def score_table(fc, which_set):
    """Score each model/version on in-stock days of one split, the same way stream 2 does."""
    s = fc[(fc.set == which_set) & (fc.oos_hours == 0)]
    rows = []
    for (model, version), g in s.groupby(['model', 'version']):
        rows.append({'model': model, 'version': version,
                     'wape': wape(g.sales, g.forecast),
                     'mae': mae(g.sales, g.forecast),
                     'bias': bias(g.sales, g.forecast),
                     'n_days': len(g)})
    return pd.DataFrame(rows)

print("Test week, in-stock days:")
score_table(fc2, 'test').round(3)

Test week, in-stock days:


,model,version,wape,mae,bias,n_days
0,lightgbm,corrected,0.350,0.415,-0.077,8285
1,lightgbm,raw,0.389,0.462,-0.211,8285
2,seasonal_naive,corrected,0.384,0.455,0.103,8285
3,seasonal_naive,raw,0.385,0.457,-0.017,8285


In [5]:
CACHE = PROC_DIR / 'stream4_cache'
CACHE.mkdir(exist_ok=True)

def cached(name, make):
    """Load saved forecasts if they exist, otherwise run the model and save right away."""
    path = CACHE / f'{name}.parquet'
    if path.exists():
        print(f"Loaded {name} from Drive")
        return pd.read_parquet(path)
    df = make()
    df.to_parquet(path, index=False)
    print(f"Saved {name} to Drive")
    return df

## Step 3: Try Chronos-2 on one product
Chronos-2 is a pretrained forecasting model from Amazon. Zero-shot means we don't train it: we give it a product's past sales and it forecasts the next days directly. We test one product first to see what goes in and what comes out before running all 2,000.

In [6]:
!pip install -q chronos-forecasting
import torch
from chronos import Chronos2Pipeline

pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# One product, history up to June 19 (7 days before the test week starts)
one_id = sample_ids[0]
hist = dd[(dd.series_id == one_id) & (dd.dt <= '2024-06-19')]
context = hist[['series_id', 'dt', 'sales']].rename(
    columns={'series_id': 'id', 'dt': 'timestamp', 'sales': 'target'})

pred = pipe.predict_df(context, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],
                       id_column='id', timestamp_column='timestamp', target='target')
print(pred.columns.tolist())
pred

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

['id', 'timestamp', 'target_name', 'predictions', '0.1', '0.5', '0.9']


,id,timestamp,target_name,predictions,0.1,0.5,0.9
0,69_257,2024-06-20,target,0.514767,0.157087,0.514767,0.953986
1,69_257,2024-06-21,target,0.512492,0.148873,0.512492,0.954392
2,69_257,2024-06-22,target,0.505079,0.152553,0.505079,0.961223
3,69_257,2024-06-23,target,0.510650,0.149854,0.510650,0.959488
4,69_257,2024-06-24,target,0.507927,0.146891,0.507927,0.952463
5,69_257,2024-06-25,target,0.504845,0.153611,0.504845,0.956414
6,69_257,2024-06-26,target,0.509187,0.154127,0.509187,0.963090


## Step 4: Chronos-2 zero-shot on raw sales
We forecast all 2,000 products for every valid and test day, following the 7-days-ahead rule: for each target day, the model only sees sales up to 7 days earlier, and we keep its 7th-day forecast. Then we score it the same way as stream 2.

In [7]:
def run_chronos(value_col, version):
    """Forecast every valid/test day exactly 7 days ahead. value_col: 'sales' (raw) or 'demand' (corrected)."""
    target_days = sorted(dd[dd.set.isin(['valid', 'test'])].dt.unique())
    out = []
    for t in target_days:
        origin = t - pd.Timedelta(days=7)
        ctx = dd[dd.dt <= origin][['series_id', 'dt', value_col]].rename(
            columns={'series_id': 'id', 'dt': 'timestamp', value_col: 'target'})
        p = pipe.predict_df(ctx, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],
                            id_column='id', timestamp_column='timestamp', target='target')
        out.append(p[p.timestamp == t])   # keep only the 7th day
    p = pd.concat(out).rename(columns={'id': 'series_id', 'timestamp': 'dt', 'predictions': 'forecast',
                                       '0.1': 'q10', '0.5': 'q50', '0.9': 'q90'})
    # Sales can't be negative
    for c in ['forecast', 'q10', 'q50', 'q90']:
        p[c] = p[c].clip(lower=0)
    # Attach the same info columns stream 2 uses (set, sales, oos_hours, category...)
    info = fc2[(fc2.model == 'seasonal_naive') & (fc2.version == 'raw')].drop(
        columns=['model', 'version', 'forecast', 'q10', 'q50', 'q90'])
    p = info.merge(p[['series_id', 'dt', 'forecast', 'q10', 'q50', 'q90']], on=['series_id', 'dt'])
    p['model'], p['version'] = 'chronos2', version
    return p

fc_chronos_raw = cached('chronos2_raw', lambda: run_chronos('sales', 'raw'))
print(fc_chronos_raw.shape, "| missing forecasts:", fc_chronos_raw.forecast.isna().sum())

compare = pd.concat([fc2, fc_chronos_raw], ignore_index=True)
score_table(compare, 'test').round(3)

Loaded chronos2_raw from Drive
(42000, 14) | missing forecasts: 0


,model,version,wape,mae,bias,n_days
0,chronos2,raw,0.332,0.393,-0.106,8285
1,lightgbm,corrected,0.350,0.415,-0.077,8285
2,lightgbm,raw,0.389,0.462,-0.211,8285
3,seasonal_naive,corrected,0.384,0.455,0.103,8285
4,seasonal_naive,raw,0.385,0.457,-0.017,8285


## Step 5: Is the result real? Validation week and corrected history
Chronos-2 beat every stream 2 model on the test week without any training. We check that it also wins on the validation week, then test whether giving it corrected demand history helps, as it did for LightGBM.

In [8]:
fc_chronos_corr = cached('chronos2_corrected', lambda: run_chronos('demand', 'corrected'))

compare = pd.concat([fc2, fc_chronos_raw, fc_chronos_corr], ignore_index=True)
print("Validation week:"); display(score_table(compare, 'valid').round(3))
print("Test week:");       display(score_table(compare, 'test').round(3))

Loaded chronos2_corrected from Drive
Validation week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.342,0.355,0.019,15821
1,chronos2,raw,0.360,0.374,-0.098,15821
2,lightgbm,corrected,0.351,0.364,0.021,15821
3,lightgbm,raw,0.365,0.379,-0.121,15821
4,seasonal_naive,corrected,0.414,0.430,0.068,15821
5,seasonal_naive,raw,0.422,0.438,-0.066,15821


Test week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.316,0.374,0.012,8285
1,chronos2,raw,0.332,0.393,-0.106,8285
2,lightgbm,corrected,0.350,0.415,-0.077,8285
3,lightgbm,raw,0.389,0.462,-0.211,8285
4,seasonal_naive,corrected,0.384,0.455,0.103,8285
5,seasonal_naive,raw,0.385,0.457,-0.017,8285


## Step 6: Where does correction help most? Then save
Stream 2 found correction helps most for products that stock out often. We check whether Chronos-2 shows the same pattern, then save the forecasts in stream 2's format for the ordering simulation (stream 5).

In [9]:
# Test-week error by stockout group (low / medium / high), raw vs corrected
test_in_stock = compare[(compare.set == 'test') & (compare.oos_hours == 0)]
by_band = (test_in_stock.groupby(['stockout_band', 'model', 'version'])
           .apply(lambda g: wape(g.sales, g.forecast))
           .unstack('version').round(3))
display(by_band)

# Save Chronos-2 forecasts in the same layout as stream 2's file
fc_stream4 = pd.concat([fc_chronos_raw, fc_chronos_corr], ignore_index=True)
fc_stream4.to_parquet(PROC_DIR / 'forecasts_stream4_v1.parquet', index=False)
print("Saved:", fc_stream4.shape, "->", PROC_DIR / 'forecasts_stream4_v1.parquet')

/tmp/ipykernel_528/3441576927.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: wape(g.sales, g.forecast))


version                       corrected    raw
stockout_band model                           
high          chronos2            0.263  0.282
              lightgbm            0.319  0.393
              seasonal_naive      0.304  0.311
low           chronos2            0.371  0.387
              lightgbm            0.389  0.396
              seasonal_naive      0.468  0.468
medium        chronos2            0.350  0.360
              lightgbm            0.364  0.377
              seasonal_naive      0.435  0.429

Saved: (84000, 14) -> /content/drive/MyDrive/MIS 382 Deep Learning and ML/Final Project/Stockout Project Data/processed/forecasts_stream4_v1.parquet


## Step 7: Does Chronos-2 use extra inputs?
Chronos-2 can take covariates: extra columns known in advance, like discounts, holidays and weather. LightGBM uses these. We test whether they help Chronos-2, using corrected history (the best setup so far). Like stream 2, we treat these values as known in advance.

In [10]:
cov_cols = ['discount', 'discount_missing', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature']
cov = pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=['series_id', 'dt'] + cov_cols)
cov['dt'] = pd.to_datetime(cov['dt'])
cov = cov[cov.series_id.isin(sample_ids)]
# Missing discounts: fill with the product's usual discount (the flag column keeps the info)
cov['discount'] = cov['discount'].fillna(cov.groupby('series_id')['discount'].transform('median')).fillna(1.0)
data_cov = dd.merge(cov, on=['series_id', 'dt'], how='left')

def run_chronos_cov(value_col, version):
    target_days = sorted(dd[dd.set.isin(['valid', 'test'])].dt.unique())
    rename = {'series_id': 'id', 'dt': 'timestamp', value_col: 'target'}
    out = []
    for t in target_days:
        origin = t - pd.Timedelta(days=7)
        ctx = data_cov[data_cov.dt <= origin][['series_id', 'dt', value_col] + cov_cols].rename(columns=rename)
        fut = data_cov[(data_cov.dt > origin) & (data_cov.dt <= t)][['series_id', 'dt'] + cov_cols].rename(columns=rename)
        p = pipe.predict_df(ctx, future_df=fut, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],
                            id_column='id', timestamp_column='timestamp', target='target')
        out.append(p[p.timestamp == t])
    p = pd.concat(out).rename(columns={'id': 'series_id', 'timestamp': 'dt', 'predictions': 'forecast',
                                       '0.1': 'q10', '0.5': 'q50', '0.9': 'q90'})
    for c in ['forecast', 'q10', 'q50', 'q90']:
        p[c] = p[c].clip(lower=0)
    info = fc2[(fc2.model == 'seasonal_naive') & (fc2.version == 'raw')].drop(
        columns=['model', 'version', 'forecast', 'q10', 'q50', 'q90'])
    p = info.merge(p[['series_id', 'dt', 'forecast', 'q10', 'q50', 'q90']], on=['series_id', 'dt'])
    p['model'], p['version'] = 'chronos2_cov', version
    return p

fc_chronos_cov = cached('chronos2_cov', lambda: run_chronos_cov('demand', 'corrected'))
compare = pd.concat([fc2, fc_chronos_raw, fc_chronos_corr, fc_chronos_cov], ignore_index=True)
for s in ['valid', 'test']:
    print(f"{s.title()} week:"); display(score_table(compare, s).round(3))

Loaded chronos2_cov from Drive
Valid week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.342,0.355,0.019,15821
1,chronos2,raw,0.360,0.374,-0.098,15821
2,chronos2_cov,corrected,0.346,0.360,0.039,15821
3,lightgbm,corrected,0.351,0.364,0.021,15821
4,lightgbm,raw,0.365,0.379,-0.121,15821
5,seasonal_naive,corrected,0.414,0.430,0.068,15821
6,seasonal_naive,raw,0.422,0.438,-0.066,15821


Test week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.316,0.374,0.012,8285
1,chronos2,raw,0.332,0.393,-0.106,8285
2,chronos2_cov,corrected,0.309,0.366,-0.010,8285
3,lightgbm,corrected,0.350,0.415,-0.077,8285
4,lightgbm,raw,0.389,0.462,-0.211,8285
5,seasonal_naive,corrected,0.384,0.455,0.103,8285
6,seasonal_naive,raw,0.385,0.457,-0.017,8285


## Step 8: TimesFM-3 zero-shot
Google's foundation model, run the same way as Chronos-2 (raw and corrected, 7 days ahead). Note: TimesFM-3 has a non-commercial license, which matters if a retailer wanted to use it.

In [11]:
!pip install -q "timesfm[torch]"
import timesfm
print("Version:", getattr(timesfm, '__version__', 'unknown'))
print([n for n in dir(timesfm) if 'TimesFM' in n or 'Config' in n])

Version: unknown
['ForecastConfig', 'TimesFM3Forecaster', 'TimesFM3Torch', 'TimesFM_2p5_200M_torch']


In [12]:
import inspect
for name in ['TimesFM3Torch', 'TimesFM3Forecaster']:
    cls = getattr(timesfm, name)
    print(f"=== {name} ===")
    print((cls.__doc__ or 'no docstring')[:1500])
    for m in ['from_pretrained', 'compile', 'forecast', 'forecast_df', '__init__']:
        if hasattr(cls, m):
            print(f"  {m}{inspect.signature(getattr(cls, m))}")
    print()
print("=== ForecastConfig ===")
print(inspect.signature(timesfm.ForecastConfig))

=== TimesFM3Torch ===
PyTorch inference-only implementation of TimesFM3.

Attributes:
  input_patch_len: The length of each input patch.
  output_patch_len: The length of the output forecast for each patch.
  quantiles: A list of quantiles to predict.
  residual_block_config: Configuration for the pre-transformer residual block.
  transformer_config: Configuration for the stacked transformers.
  use_variate_attention: Whether to use variate attention.
  value_clip: Absolute value to clip input values to.
  use_stitching: Whether to use stitching for predictions.
  use_linear_detrending: Whether to apply linear detrending on context.
  linear_detrending_threshold: Ratio threshold for applying linear detrending.
  use_iterative_cpm_revin: Whether to use iterative RevIN refinement.
  use_frozen_running_stats: Whether running stats freeze at context boundary.

  from_pretrained(pretrained_model_name_or_path: str | pathlib._local.Path, *, force_download: bool = False, token: str | bool | No

In [13]:
del pipe; import gc; gc.collect(); torch.cuda.empty_cache()
tfm = timesfm.TimesFM3Forecaster.from_pretrained(device='cuda')

methods = [m for m in dir(tfm) if not m.startswith('_') and callable(getattr(tfm, m))]
print(methods)
for m in methods:
    if any(k in m.lower() for k in ['forecast', 'predict']):
        print(f"\n{m}{inspect.signature(getattr(tfm, m))}")
        print((getattr(tfm, m).__doc__ or '')[:1200])

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.32GB            

model.safetensors: downloading bytes:           |  0.00B            

['from_pretrained', 'model', 'predict', 'predict_batch']

predict(context: 'np.ndarray', horizon: 'int', past_only_covariates: 'np.ndarray | None' = None, past_future_covariates: 'np.ndarray | None' = None, ts_id: 'str | None' = None, return_quantiles: 'bool' = False, use_symmetric_averaging: 'bool' = False, make_positive: 'bool' = False, sort_quantiles: 'bool' = True, use_znorm: 'bool' = False, padding_mode: 'str' = 'none') -> 'ForecastOutput'
Convenience wrapper: runs inference on a single time series.

predict_batch(contexts: 'list[np.ndarray]', horizon: 'int', past_only_covariates: 'list[np.ndarray | None] | None' = None, past_future_covariates: 'list[np.ndarray | None] | None' = None, ts_ids: 'list[str] | None' = None, return_quantiles: 'bool' = False, use_symmetric_averaging: 'bool' = False, make_positive: 'bool' = False, sort_quantiles: 'bool' = True, use_znorm: 'bool' = False, padding_mode: 'str' = 'none') -> 'Iterator[ForecastOutput]'
Runs inference on a batch of time series w

In [14]:
import numpy as np
one = dd[(dd.series_id == one_id) & (dd.dt <= '2024-06-19')].sort_values('dt')
out = tfm.predict(one['sales'].to_numpy(np.float32), horizon=7,
                  return_quantiles=True, make_positive=True)

print(type(out))
print({k: getattr(out, k).shape if hasattr(getattr(out, k), 'shape') else getattr(out, k)
       for k in dir(out) if not k.startswith('_') and not callable(getattr(out, k))})
print("Model quantiles:", getattr(tfm.model, 'quantiles', 'not found'))

<class 'timesfm3.torch.timesfm3_forecaster.ForecastOutput'>
{'forecast': (7,), 'quantiles': (7, 9), 'ts_id': None}
Model quantiles: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


In [15]:
def run_timesfm(value_col, version):
    """Same 7-days-ahead setup as Chronos-2."""
    target_days = sorted(dd[dd.set.isin(['valid', 'test'])].dt.unique())
    d = dd.sort_values(['series_id', 'dt'])
    out = []
    for t in target_days:
        origin = t - pd.Timedelta(days=7)
        hist = d[d.dt <= origin]
        ids, contexts = zip(*[(sid, g[value_col].to_numpy(np.float32))
                              for sid, g in hist.groupby('series_id')])
        results = list(tfm.predict_batch(list(contexts), horizon=7,
                                         return_quantiles=True, make_positive=True))
        out.append(pd.DataFrame({
            'series_id': ids, 'dt': t,
            'forecast': [r.forecast[6] for r in results],      # 7th day
            'q10': [r.quantiles[6, 0] for r in results],
            'q50': [r.quantiles[6, 4] for r in results],
            'q90': [r.quantiles[6, 8] for r in results]}))
    p = pd.concat(out)
    for c in ['forecast', 'q10', 'q50', 'q90']:
        p[c] = p[c].clip(lower=0)
    info = fc2[(fc2.model == 'seasonal_naive') & (fc2.version == 'raw')].drop(
        columns=['model', 'version', 'forecast', 'q10', 'q50', 'q90'])
    p = info.merge(p, on=['series_id', 'dt'])
    p['model'], p['version'] = 'timesfm3', version
    return p

fc_tfm_raw = cached('timesfm3_raw', lambda: run_timesfm('sales', 'raw'))
fc_tfm_corr = cached('timesfm3_corrected', lambda: run_timesfm('demand', 'corrected'))
print(fc_tfm_raw.shape, fc_tfm_corr.shape)

compare = pd.concat([fc2, fc_chronos_raw, fc_chronos_corr, fc_chronos_cov,
                     fc_tfm_raw, fc_tfm_corr], ignore_index=True)
for s in ['valid', 'test']:
    print(f"{s.title()} week:"); display(score_table(compare, s).round(3))

Loaded timesfm3_raw from Drive
Loaded timesfm3_corrected from Drive
(42000, 14) (42000, 14)
Valid week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.342,0.355,0.019,15821
1,chronos2,raw,0.360,0.374,-0.098,15821
2,chronos2_cov,corrected,0.346,0.360,0.039,15821
3,lightgbm,corrected,0.351,0.364,0.021,15821
4,lightgbm,raw,0.365,0.379,-0.121,15821
5,seasonal_naive,corrected,0.414,0.430,0.068,15821
6,seasonal_naive,raw,0.422,0.438,-0.066,15821
7,timesfm3,corrected,0.346,0.360,0.018,15821
8,timesfm3,raw,0.360,0.374,-0.098,15821


Test week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.316,0.374,0.012,8285
1,chronos2,raw,0.332,0.393,-0.106,8285
2,chronos2_cov,corrected,0.309,0.366,-0.010,8285
3,lightgbm,corrected,0.350,0.415,-0.077,8285
4,lightgbm,raw,0.389,0.462,-0.211,8285
5,seasonal_naive,corrected,0.384,0.455,0.103,8285
6,seasonal_naive,raw,0.385,0.457,-0.017,8285
7,timesfm3,corrected,0.317,0.375,0.001,8285
8,timesfm3,raw,0.331,0.392,-0.098,8285


In [16]:
a = fc_chronos_raw.set_index(['series_id', 'dt']).forecast
b = fc_tfm_raw.set_index(['series_id', 'dt']).forecast
print("Identical forecasts:", np.allclose(a.sort_index(), b.sort_index()))
print("Correlation:", round(a.sort_index().corr(b.sort_index()), 3))

Identical forecasts: False
Correlation: 0.992


## Step 9: Is it luck? 2,000 random re-draws
Like stream 2, we re-score the test week 2,000 times on random samples of products. If a model wins in nearly every draw, the gap is real, not luck.

In [17]:
def per_product(fc, model, version):
    """Total absolute error and total sales per product, test week, in-stock days."""
    s = fc[(fc.set == 'test') & (fc.oos_hours == 0) & (fc.model == model) & (fc.version == version)]
    return s.assign(err=(s.sales - s.forecast).abs()).groupby('series_id')[['err', 'sales']].sum()

ids = per_product(compare, 'chronos2', 'raw').index
rng = np.random.default_rng(42)
draws = rng.integers(0, len(ids), size=(2000, len(ids)))   # 2,000 random samples of products

def boot_wape(model, version):
    t = per_product(compare, model, version).reindex(ids).fillna(0)
    return t.err.values[draws].sum(1) / t.sales.values[draws].sum(1)

pairs = [(('chronos2', 'corrected'), ('chronos2', 'raw')),
         (('timesfm3', 'corrected'), ('timesfm3', 'raw')),
         (('chronos2', 'corrected'), ('lightgbm', 'corrected')),
         (('chronos2', 'raw'),       ('lightgbm', 'raw')),
         (('chronos2', 'corrected'), ('timesfm3', 'corrected'))]

rows = []
for a, b in pairs:
    wa, wb = boot_wape(*a), boot_wape(*b)
    rows.append({'model A': ' '.join(a), 'model B': ' '.join(b),
                 'A wins (% of draws)': round((wa < wb).mean() * 100, 1),
                 'avg gap (B - A)': round((wb - wa).mean(), 3)})
pd.DataFrame(rows)

,model A,model B,A wins (% of draws),avg gap (B - A)
0,chronos2 corrected,chronos2 raw,100.0,0.016
1,timesfm3 corrected,timesfm3 raw,100.0,0.014
2,chronos2 corrected,lightgbm corrected,100.0,0.035
3,chronos2 raw,lightgbm raw,100.0,0.057
4,chronos2 corrected,timesfm3 corrected,70.2,0.001


## Step 10: Fine-tune Chronos-2 with LoRA
Zero-shot Chronos-2 has never seen grocery data. LoRA fine-tuning trains small add-on weights while keeping the pretrained model frozen, so it adapts cheaply. Fine-tuning data stops on Jun 4, the day before the earliest forecast origin, so no forecast day leaks into training. We fine-tune on raw and corrected history to check that correction still helps after fine-tuning.

In [18]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [19]:
import gc
from chronos import Chronos2Pipeline

# Free the GPU (TimesFM is no longer needed, its results are saved)
if 'tfm' in globals(): del tfm
gc.collect(); torch.cuda.empty_cache()

CUT = pd.Timestamp('2024-06-04')

def lora_forecasts(value_col, version, steps=1000):
    """Fine-tune Chronos-2 on history up to Jun 4, then forecast 7 days ahead like before."""
    global pipe
    base = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
    series = [g.sort_values('dt')[value_col].to_numpy(np.float32)
              for _, g in dd[dd.dt <= CUT].groupby('series_id')]
    pipe = base.fit(series, prediction_length=7, finetune_mode='lora', learning_rate=1e-5,
                    num_steps=steps, batch_size=64, context_length=64,
                    output_dir=f'/content/ft_{value_col}')
    p = run_chronos(value_col, version)     # same forecasting loop as zero-shot
    p['model'] = 'chronos2_lora'
    return p

fc_lora_raw  = cached('chronos2_lora_raw',       lambda: lora_forecasts('sales', 'raw'))
fc_lora_corr = cached('chronos2_lora_corrected', lambda: lora_forecasts('demand', 'corrected'))

compare = pd.concat([fc2, fc_chronos_raw, fc_chronos_corr, fc_lora_raw, fc_lora_corr], ignore_index=True)
for s in ['valid', 'test']:
    print(f"{s.title()} week:"); display(score_table(compare, s).round(3))

for a, b in [(('chronos2_lora', 'corrected'), ('chronos2', 'corrected')),
             (('chronos2_lora', 'corrected'), ('chronos2_lora', 'raw'))]:
    wa, wb = boot_wape(*a), boot_wape(*b)
    print(' '.join(a), 'beats', ' '.join(b), f'in {(wa < wb).mean()*100:.1f}% of draws, avg gap {(wb - wa).mean():.3f}')

Loaded chronos2_lora_raw from Drive
Loaded chronos2_lora_corrected from Drive
Valid week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.342,0.355,0.019,15821
1,chronos2,raw,0.360,0.374,-0.098,15821
2,chronos2_lora,corrected,0.339,0.352,0.015,15821
3,chronos2_lora,raw,0.353,0.367,-0.081,15821
4,lightgbm,corrected,0.351,0.364,0.021,15821
5,lightgbm,raw,0.365,0.379,-0.121,15821
6,seasonal_naive,corrected,0.414,0.430,0.068,15821
7,seasonal_naive,raw,0.422,0.438,-0.066,15821


Test week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.316,0.374,0.012,8285
1,chronos2,raw,0.332,0.393,-0.106,8285
2,chronos2_lora,corrected,0.314,0.373,0.002,8285
3,chronos2_lora,raw,0.324,0.384,-0.091,8285
4,lightgbm,corrected,0.350,0.415,-0.077,8285
5,lightgbm,raw,0.389,0.462,-0.211,8285
6,seasonal_naive,corrected,0.384,0.455,0.103,8285
7,seasonal_naive,raw,0.385,0.457,-0.017,8285


chronos2_lora corrected beats chronos2 corrected in 87.6% of draws, avg gap 0.001
chronos2_lora corrected beats chronos2_lora raw in 100.0% of draws, avg gap 0.010


## Step 11: Can a network learn from stockouts directly? (censored loss)
Instead of correcting the data first, we train a neural network on raw sales and tell it which days had stockouts. On a stockout day, recorded sales are only a lower bound on demand, so the loss penalizes forecasts below that day's sales but not above them (a censored loss). We compare three versions of the same network: raw sales, corrected demand (two-step, like the rest of the project), and censored (end-to-end, no correction).

The network is a small GRU (a recurrent network) that reads the last 28 days ending 7 days before the target day, plus a category embedding and the day's discount, holiday and weather. It uses dropout and early stopping on the validation week, and we average 3 random seeds.

In [20]:
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore', message='Mean of empty slice')
dev = 'cuda'
W = 28   # days of history the network reads

# ---- Arrange the data as products x days grids
cols = ['discount', 'discount_missing', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'first_category_id', 'day_of_week']
cv = pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=['series_id', 'dt'] + cols)
cv = cv[cv.series_id.isin(sample_ids)]; cv['dt'] = pd.to_datetime(cv['dt'])
d = dd.merge(cv, on=['series_id', 'dt'], how='left').sort_values(['series_id', 'dt']).reset_index(drop=True)
d['discount'] = d.discount.fillna(d.groupby('series_id').discount.transform('median')).fillna(1.0)
pids = d.series_id.unique(); S, T = len(pids), d.dt.nunique()
grid = lambda c: d[c].to_numpy('float32').reshape(S, T)
sales, demand, oos = grid('sales'), grid('demand'), grid('oos_hours')
dow = d.day_of_week.to_numpy().reshape(S, T)
X_day = np.stack([grid('discount'), grid('discount_missing'), grid('holiday_flag'), grid('activity_flag'),
                  grid('precpt') / 10, (grid('avg_temperature') - 22) / 4], -1)
cat = pd.factorize(d.groupby('series_id', sort=False).first_category_id.first())[0]
sets = d.set.to_numpy().reshape(S, T)[0]; dates = np.sort(d.dt.unique())
train_t, valid_t, test_t = [np.where(sets == s)[0] for s in ['train', 'valid', 'test']]

def make(version):
    """Inputs and targets for one version: 'raw', 'corrected' or 'censored'."""
    val = demand if version == 'corrected' else sales
    if version == 'censored':   # scale from in-stock training days only
        m = np.where(oos[:, train_t] == 0, val[:, train_t], np.nan)
        scale = np.where(np.isnan(np.nanmean(m, 1)), val[:, train_t].mean(1), np.nanmean(m, 1))
    else:
        scale = val[:, train_t].mean(1)
    scale = np.maximum(scale, 0.01).astype('float32')
    def rows(ts):
        si, ti = [a.ravel() for a in np.meshgrid(np.arange(S), ts, indexing='ij')]
        win = ti[:, None] - 7 - W + 1 + np.arange(W)            # the 28 days ending 7 days before the target
        feats = [val[si[:, None], win] / scale[si, None]]
        if version == 'censored':
            feats.append(oos[si[:, None], win] / 16)           # which past days had empty shelves
        xs = np.stack(feats, -1).astype('float32')
        xd = np.concatenate([X_day[si, ti], np.eye(7, dtype='float32')[dow[si, ti]]], 1)
        y = (val[si, ti] / scale[si]).astype('float32')
        cens = ((oos[si, ti] > 0) & (version == 'censored')).astype('float32')
        return [torch.tensor(a).to(dev) for a in (xs, xd, cat[si], y, cens)], si, ti
    return rows(train_t[train_t >= 7 + W - 1]), rows(valid_t), rows(test_t), scale

class Net(nn.Module):
    def __init__(self, n_ch, n_cat):
        super().__init__()
        self.gru = nn.GRU(n_ch, 32, batch_first=True)
        self.emb = nn.Embedding(n_cat, 8)
        self.head = nn.Sequential(nn.Linear(32 + 8 + 13, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 1))
    def forward(self, xs, xd, c):
        _, h = self.gru(xs)
        return nn.functional.softplus(self.head(torch.cat([h[-1], self.emb(c), xd], 1))).squeeze(-1)

def loss_fn(pred, y, cens):
    err = y - pred
    return torch.where(cens > 0, torch.clamp(err, min=0), err.abs()).mean()   # censored day: only penalize forecasts below sales

def fit_predict(version, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    (tr, _, _), (va, vsi, vti), (te, tsi, tti), scale = make(version)
    net = Net(tr[0].shape[-1], cat.max() + 1).to(dev); opt = torch.optim.Adam(net.parameters(), 2e-3)
    v_ok = oos[vsi, vti] == 0
    best, state, bad = 1e9, None, 0
    for ep in range(40):
        net.train(); perm = torch.randperm(len(tr[3]), device=dev)
        for i in range(0, len(perm), 1024):
            b = perm[i:i + 1024]
            loss = loss_fn(net(tr[0][b], tr[1][b], tr[2][b]), tr[3][b], tr[4][b])
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pv = net(*va[:3]).cpu().numpy() * scale[vsi]
        score = wape(sales[vsi, vti][v_ok], pv[v_ok])                          # early stopping on valid, in-stock days
        if score < best - 1e-4: best, state, bad = score, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(state); net.eval()
    with torch.no_grad():
        pv, pt = net(*va[:3]).cpu().numpy() * scale[vsi], net(*te[:3]).cpu().numpy() * scale[tsi]
    return pd.DataFrame({'series_id': np.r_[pids[vsi], pids[tsi]], 'dt': np.r_[dates[vti], dates[tti]], 'forecast': np.r_[pv, pt]})

def run_gru(version):
    outs = [fit_predict(version, s) for s in (0, 1, 2)]                       # average 3 seeds
    p = outs[0].copy(); p['forecast'] = np.mean([o.forecast.values for o in outs], 0)
    info = fc2[(fc2.model == 'seasonal_naive') & (fc2.version == 'raw')].drop(columns=['model', 'version', 'forecast', 'q10', 'q50', 'q90'])
    p = info.merge(p, on=['series_id', 'dt']); p['model'], p['version'] = 'gru', version
    return p

fc_gru = {v: cached(f'gru_{v}', lambda v=v: run_gru(v)) for v in ['raw', 'corrected', 'censored']}
compare = pd.concat([fc2, fc_chronos_corr] + list(fc_gru.values()), ignore_index=True)
for s in ['valid', 'test']:
    print(f"{s.title()} week:"); display(score_table(compare, s).round(3))
for a, b in [(('gru', 'censored'), ('gru', 'corrected')), (('gru', 'censored'), ('lightgbm', 'corrected'))]:
    wa, wb = boot_wape(*a), boot_wape(*b)
    print(' '.join(a), 'beats', ' '.join(b), f'in {(wa < wb).mean()*100:.1f}% of draws, avg gap {(wb - wa).mean():.3f}')

Loaded gru_raw from Drive
Loaded gru_corrected from Drive
Loaded gru_censored from Drive
Valid week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.342,0.355,0.019,15821
1,gru,censored,0.348,0.361,-0.031,15821
2,gru,corrected,0.353,0.367,0.004,15821
3,gru,raw,0.366,0.380,-0.095,15821
4,lightgbm,corrected,0.351,0.364,0.021,15821
5,lightgbm,raw,0.365,0.379,-0.121,15821
6,seasonal_naive,corrected,0.414,0.430,0.068,15821
7,seasonal_naive,raw,0.422,0.438,-0.066,15821


Test week:


,model,version,wape,mae,bias,n_days
0,chronos2,corrected,0.316,0.374,0.012,8285
1,gru,censored,0.336,0.399,-0.100,8285
2,gru,corrected,0.345,0.409,-0.072,8285
3,gru,raw,0.363,0.431,-0.158,8285
4,lightgbm,corrected,0.350,0.415,-0.077,8285
5,lightgbm,raw,0.389,0.462,-0.211,8285
6,seasonal_naive,corrected,0.384,0.455,0.103,8285
7,seasonal_naive,raw,0.385,0.457,-0.017,8285


gru censored beats gru corrected in 98.4% of draws, avg gap 0.008
gru censored beats lightgbm corrected in 100.0% of draws, avg gap 0.014


## Step 12: Rolling backtest (does it hold beyond one week?)
So far every result comes from one test week (Jun 26 to Jul 2). We repeat the forecasts for four earlier weeks (May 15 to Jun 11) plus the official week, each day forecast exactly 7 days ahead using only data up to 7 days before it. We report each model's error per week, the average, the spread, and how often correction wins week by week.

Note: missed demand v1 was filled by models trained on all training days, so in earlier weeks the corrected history comes from a fill model that saw slightly later data. The effect should be small, since it only touches the filled-in hours, not the days being scored.

In [21]:
from chronos import Chronos2Pipeline
# Step 10 left the fine-tuned model in `pipe`, so reload the original zero-shot Chronos-2
pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

WEEKS = {'W1 May 15-21': '2024-05-15', 'W2 May 22-28': '2024-05-22', 'W3 May 29-Jun 4': '2024-05-29',
         'W4 Jun 5-11': '2024-06-05', 'Test Jun 26-Jul 2': '2024-06-26'}
bt_days = [(w, pd.Timestamp(s) + pd.Timedelta(days=i)) for w, s in WEEKS.items() for i in range(7)]
bands = fc2[['series_id', 'stockout_band']].drop_duplicates()
base_bt = dd[dd.dt.isin([t for _, t in bt_days])][['series_id', 'dt', 'sales', 'demand', 'oos_hours']].merge(bands, on='series_id')
base_bt['week'] = base_bt.dt.map({t: w for w, t in bt_days})

def snaive_bt(value_col, version):
    """Seasonal naive: same weekday last week (exactly 7 days back)."""
    lag = dd[['series_id', 'dt', value_col]].copy(); lag['dt'] = lag.dt + pd.Timedelta(days=7)
    p = base_bt.merge(lag.rename(columns={value_col: 'forecast'}), on=['series_id', 'dt'])
    p['model'], p['version'] = 'seasonal_naive', version
    return p

def chronos_bt(value_col, version):
    """Zero-shot Chronos-2, each backtest day forecast 7 days ahead."""
    out = []
    for _, t in bt_days:
        ctx = dd[dd.dt <= t - pd.Timedelta(days=7)][['series_id', 'dt', value_col]].rename(
            columns={'series_id': 'id', 'dt': 'timestamp', value_col: 'target'})
        p = pipe.predict_df(ctx, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],
                            id_column='id', timestamp_column='timestamp', target='target')
        out.append(p[p.timestamp == t])
    p = pd.concat(out).rename(columns={'id': 'series_id', 'timestamp': 'dt', 'predictions': 'forecast',
                                       '0.1': 'q10', '0.5': 'q50', '0.9': 'q90'})
    p = base_bt.merge(p[['series_id', 'dt', 'forecast', 'q10', 'q50', 'q90']], on=['series_id', 'dt'])
    for c in ['forecast', 'q10', 'q50', 'q90']: p[c] = p[c].clip(lower=0)
    p['model'], p['version'] = 'chronos2', version
    return p

bt = pd.concat([cached('bt_snaive_raw', lambda: snaive_bt('sales', 'raw')),
                cached('bt_snaive_corrected', lambda: snaive_bt('demand', 'corrected')),
                cached('bt_chronos2_raw', lambda: chronos_bt('sales', 'raw')),
                cached('bt_chronos2_corrected', lambda: chronos_bt('demand', 'corrected'))], ignore_index=True)

def bt_table(fc, metric):
    """One row per model/version, one column per week, plus average and spread. In-stock days only."""
    s = fc[fc.oos_hours == 0]
    t = s.groupby(['model', 'version', 'week']).apply(lambda g: metric(g.sales, g.forecast), include_groups=False).unstack('week')[list(WEEKS)]
    t['average'] = t[list(WEEKS)].mean(1)
    t['spread (max-min)'] = t[list(WEEKS)].max(1) - t[list(WEEKS)].min(1)
    return t.round(3)

print("WAPE by week (lower is better):"); w = bt_table(bt, wape); display(w)
print("Bias by week (negative = under-forecasting):"); display(bt_table(bt, bias))
for m in ['seasonal_naive', 'chronos2']:
    wins = (w.loc[(m, 'corrected'), list(WEEKS)] < w.loc[(m, 'raw'), list(WEEKS)]).sum()
    print(f"{m}: corrected beats raw in {wins} of {len(WEEKS)} weeks")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loaded bt_snaive_raw from Drive
Loaded bt_snaive_corrected from Drive
Loaded bt_chronos2_raw from Drive
Loaded bt_chronos2_corrected from Drive
WAPE by week (lower is better):


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.366         0.352            0.352   
               raw               0.369         0.365            0.348   
seasonal_naive corrected         0.450         0.414            0.430   
               raw               0.451         0.417            0.425   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  \
model          version                                              
chronos2       corrected        0.375              0.316    0.352   
               raw              0.367              0.332    0.356   
seasonal_naive corrected        0.446              0.384    0.425   
               raw              0.439              0.385    0.424   

week                      spread (max-min)  
model          version                      
chronos2       corrected             0.059  
               raw                   0.038  
seasonal_naive corrected             0.067  
               raw                   0.065

Bias by week (negative = under-forecasting):


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.102         0.079            0.141   
               raw              -0.053        -0.064           -0.011   
seasonal_naive corrected         0.181         0.119            0.154   
               raw               0.036        -0.002            0.030   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  \
model          version                                              
chronos2       corrected        0.128              0.012    0.092   
               raw              0.002             -0.106   -0.047   
seasonal_naive corrected        0.134              0.103    0.138   
               raw              0.016             -0.017    0.013   

week                      spread (max-min)  
model          version                      
chronos2       corrected             0.129  
               raw                   0.108  
seasonal_naive corrected             0.078  
               raw                   0.053

seasonal_naive: corrected beats raw in 3 of 5 weeks
chronos2: corrected beats raw in 3 of 5 weeks


In [22]:
# Are fully in-stock days quieter than average? (per week, 2,000 products)
chk = dd[dd.dt.isin([t for _, t in bt_days])].copy()
chk['week'] = chk.dt.map({t: w for w, t in bt_days})
chk['prod_mean'] = chk.groupby('series_id').demand.transform('mean')      # each product's average daily demand in that period
chk['rel'] = chk.demand / chk.prod_mean.clip(lower=1e-6)                  # demand relative to the product's own average
out = chk.groupby(['week', chk.oos_hours == 0]).rel.mean().unstack().rename(columns={True: 'in-stock days', False: 'stockout days'})
out['in-stock vs average'] = out['in-stock days'] - 1
display(out.loc[list(WEEKS)].round(3))

oos_hours,stockout days,in-stock days,in-stock vs average
week,,,
W1 May 15-21,1.150,0.847,-0.153
W2 May 22-28,1.154,0.854,-0.146
W3 May 29-Jun 4,1.156,0.832,-0.168
W4 Jun 5-11,1.196,0.844,-0.156
Test Jun 26-Jul 2,1.338,0.969,-0.031


## Step 12b: Fair evaluation (scores that use every day)
Fully in-stock days are 15 to 17% quieter than average in most weeks, because stockouts happen on busy days. Scoring only on in-stock days therefore favors forecasts that run low, and is biased against correction. We add two scores that use every day and don't depend on our missed demand estimate:
- **Observed-hours error:** on every day, compare the forecast with sales in the hours that were in stock, including the busy hours before a stockout. Each daily forecast is split into hours using the product's normal hourly pattern (from in-stock hours up to May 7, before the first backtest forecast).
- **Censored-aware error:** on in-stock days, the usual error. On stockout days, true demand was at least recorded sales, so only forecasts below recorded sales count as errors.

In [23]:
# ---- Hourly data: which hours were in stock, and the product's normal hourly pattern
hr = pd.read_parquet(PROC_DIR / 'hourly_demand_v1.parquet', columns=['series_id', 'dt', 'hour', 'sales', 'out_of_stock'])
hr['dt'] = pd.to_datetime(hr['dt'])
hr['weekend'] = (hr.dt.dt.dayofweek >= 5).astype('int8')

src = hr[(hr.dt <= '2024-05-07') & (hr.out_of_stock == 0)]          # pattern from before the first backtest forecast
prof = src.pivot_table(index=['series_id', 'weekend'], columns='hour', values='sales', aggfunc='mean')
prof = prof.reindex(columns=range(6, 22)).fillna(0)
tot = prof.sum(1)
prof = prof.div(tot.where(tot > 0), axis=0).fillna(1 / 16)            # share of the day's sales in each hour
prof = prof.stack().rename('share').reset_index()

hr = hr.merge(prof, on=['series_id', 'weekend', 'hour'], how='left')
hr['share'] = hr.share.fillna(1 / 16)
obs = (hr[hr.out_of_stock == 0].groupby(['series_id', 'dt'])
       .agg(obs_sales=('sales', 'sum'), obs_share=('share', 'sum')).reset_index())
del hr, src

def fair_table(fc):
    """Per model/version/week: in-stock WAPE (old), observed-hours WAPE and bias, censored-aware WAPE, all days."""
    f = fc.merge(obs, on=['series_id', 'dt'], how='left').fillna({'obs_sales': 0, 'obs_share': 0})
    f['f_obs'] = f.forecast * f.obs_share                              # forecast for the in-stock hours only
    stock = f.oos_hours == 0
    f['cens_err'] = np.where(stock, (f.sales - f.forecast).abs(), (f.sales - f.forecast).clip(lower=0))
    rows = []
    for (m, v, w), g in f.groupby(['model', 'version', 'week']):
        s = g[g.oos_hours == 0]
        rows.append({'model': m, 'version': v, 'week': w,
                     'in-stock WAPE (old)': wape(s.sales, s.forecast),
                     'observed-hours WAPE': (g.obs_sales - g.f_obs).abs().sum() / g.obs_sales.sum(),
                     'observed-hours bias': (g.f_obs - g.obs_sales).sum() / g.obs_sales.sum(),
                     'censored-aware WAPE': g.cens_err.sum() / g.sales.sum()})
    return pd.DataFrame(rows)

ft = fair_table(bt)
for metric in ['in-stock WAPE (old)', 'observed-hours WAPE', 'observed-hours bias', 'censored-aware WAPE']:
    t = ft.pivot_table(index=['model', 'version'], columns='week', values=metric)[list(WEEKS)]
    t['average'] = t.mean(1)
    print(metric); display(t.round(3))
    if 'WAPE' in metric:
        for m in ['seasonal_naive', 'chronos2']:
            wins = (t.loc[(m, 'corrected'), list(WEEKS)] < t.loc[(m, 'raw'), list(WEEKS)]).sum()
            print(f"  {m}: corrected beats raw in {wins} of {len(WEEKS)} weeks")

in-stock WAPE (old)


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.366         0.352            0.352   
               raw               0.369         0.365            0.348   
seasonal_naive corrected         0.450         0.414            0.430   
               raw               0.451         0.417            0.425   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.375              0.316    0.352  
               raw              0.367              0.332    0.356  
seasonal_naive corrected        0.446              0.384    0.425  
               raw              0.439              0.385    0.424

  seasonal_naive: corrected beats raw in 3 of 5 weeks
  chronos2: corrected beats raw in 3 of 5 weeks
observed-hours WAPE


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.344         0.328            0.335   
               raw               0.385         0.372            0.355   
seasonal_naive corrected         0.405         0.379            0.397   
               raw               0.436         0.404            0.410   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.361              0.326    0.339  
               raw              0.375              0.359    0.369  
seasonal_naive corrected        0.416              0.372    0.394  
               raw              0.434              0.393    0.415

  seasonal_naive: corrected beats raw in 5 of 5 weeks
  chronos2: corrected beats raw in 5 of 5 weeks
observed-hours bias


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected        -0.027        -0.026            0.027   
               raw              -0.184        -0.174           -0.120   
seasonal_naive corrected         0.037         0.010            0.035   
               raw              -0.114        -0.119           -0.090   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected       -0.015             -0.087   -0.026  
               raw             -0.132             -0.200   -0.162  
seasonal_naive corrected       -0.017             -0.006    0.012  
               raw             -0.131             -0.126   -0.116

censored-aware WAPE


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.267         0.253            0.260   
               raw               0.319         0.307            0.291   
seasonal_naive corrected         0.316         0.295            0.318   
               raw               0.361         0.334            0.345   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.276              0.253    0.262  
               raw              0.302              0.294    0.303  
seasonal_naive corrected        0.329              0.289    0.310  
               raw              0.359              0.322    0.344

  seasonal_naive: corrected beats raw in 5 of 5 weeks
  chronos2: corrected beats raw in 5 of 5 weeks


## Step 12c: Backtest for the trained models (LightGBM and GRU)
Same 5 weeks and 7-days-ahead rule. Trained models are retrained for each week using only data known before that week's first forecast: training targets up to 14 days before the week, the following 7 days for early stopping. LightGBM uses stream 2's features and settings; the product-level average is recomputed from data before each cutoff. The GRU reads 14 days of history here (28 left too few training days for the earliest week). Everything is scored with in-stock, observed-hours and censored-aware error.

In [24]:
import lightgbm as lgb

# ---------------- LightGBM (stream 2's features and settings), retrained per week ----------------
lgb_cols = ['series_id', 'dt', 'day_of_week', 'discount', 'discount_missing', 'holiday_flag', 'activity_flag',
            'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'first_category_id']
lctx = pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=lgb_cols)
lctx['dt'] = pd.to_datetime(lctx['dt']); lctx = lctx[lctx.series_id.isin(sample_ids)]
lbase = dd.merge(lctx, on=['series_id', 'dt'], how='left').sort_values(['series_id', 'dt']).reset_index(drop=True)
lbase['first_category_id'] = lbase.first_category_id.astype('int64')
LGB_FEATS = ['lag7', 'lag14', 'lag21', 'roll7', 'roll14', 'dow_avg3', 'train_mean', 'day_of_week', 'discount',
             'discount_missing', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity',
             'avg_wind_level', 'first_category_id']
LGB_PARAMS = dict(objective='tweedie', n_estimators=1500, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def lgb_features(value_col, cutoff):
    d = lbase.copy(); d['target'] = d[value_col]; g = d.groupby('series_id')['target']
    d['lag7'], d['lag14'], d['lag21'] = g.shift(7), g.shift(14), g.shift(21)
    d['roll7'] = g.transform(lambda s: s.shift(7).rolling(7, min_periods=3).mean())
    d['roll14'] = g.transform(lambda s: s.shift(7).rolling(14, min_periods=7).mean())
    d['dow_avg3'] = d[['lag7', 'lag14', 'lag21']].mean(axis=1)
    d['train_mean'] = d.series_id.map(d[d.dt <= cutoff].groupby('series_id').target.mean())   # only data before the cutoff
    return d

def lgb_bt(value_col, version):
    out = []
    for s in WEEKS.values():
        s = pd.Timestamp(s); cutoff = s - pd.Timedelta(days=7)          # last day known when forecasting the week's first day
        d = lgb_features(value_col, cutoff)
        tr = d[(d.dt <= cutoff - pd.Timedelta(days=7)) & d.lag7.notna()]
        va = d[(d.dt > cutoff - pd.Timedelta(days=7)) & (d.dt <= cutoff)]  # early stopping only
        te = d[(d.dt >= s) & (d.dt < s + pd.Timedelta(days=7))]
        m = lgb.LGBMRegressor(**LGB_PARAMS).fit(tr[LGB_FEATS], tr.target, eval_set=[(va[LGB_FEATS], va.target)],
                                                categorical_feature=['first_category_id'],
                                                callbacks=[lgb.early_stopping(50, verbose=False)])
        out.append(te[['series_id', 'dt']].assign(forecast=np.clip(m.predict(te[LGB_FEATS]), 0, None)))
    p = base_bt.merge(pd.concat(out), on=['series_id', 'dt']); p['model'], p['version'] = 'lightgbm', version
    return p

# ---------------- GRU (same network as Step 11), retrained per week, 14 days of history ----------------
W_BT = 14
day_idx = {t: i for i, t in enumerate(pd.to_datetime(dates))}

def gru_fit_week(version, s, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    t0 = day_idx[pd.Timestamp(s)]; cut = t0 - 7
    tr_ts, va_ts, te_ts = np.arange(7 + W_BT - 1, cut - 6), np.arange(cut - 6, cut + 1), np.arange(t0, t0 + 7)
    val = demand if version == 'corrected' else sales
    known = np.arange(0, cut + 1)
    if version == 'censored':
        mm = np.where(oos[:, known] == 0, val[:, known], np.nan); sc = np.nanmean(mm, 1)
        scale = np.where(np.isnan(sc), val[:, known].mean(1), sc)
    else:
        scale = val[:, known].mean(1)
    scale = np.maximum(scale, 0.01).astype('float32')
    def rows(ts):
        si, ti = [a.ravel() for a in np.meshgrid(np.arange(S), ts, indexing='ij')]
        win = ti[:, None] - 7 - W_BT + 1 + np.arange(W_BT)
        feats = [val[si[:, None], win] / scale[si, None]]
        if version == 'censored': feats.append(oos[si[:, None], win] / 16)
        xs = np.stack(feats, -1).astype('float32')
        xd = np.concatenate([X_day[si, ti], np.eye(7, dtype='float32')[dow[si, ti]]], 1)
        y = (val[si, ti] / scale[si]).astype('float32')
        cens = ((oos[si, ti] > 0) & (version == 'censored')).astype('float32')
        return [torch.tensor(a).to(dev) for a in (xs, xd, cat[si], y, cens)], si, ti
    (tr, _, _), (va, vsi, vti), (te, tsi, tti) = rows(tr_ts), rows(va_ts), rows(te_ts)
    net = Net(tr[0].shape[-1], cat.max() + 1).to(dev); opt = torch.optim.Adam(net.parameters(), 2e-3)
    v_ok = oos[vsi, vti] == 0; best, state, bad = 1e9, None, 0
    for ep in range(40):
        net.train(); perm = torch.randperm(len(tr[3]), device=dev)
        for i in range(0, len(perm), 1024):
            b = perm[i:i + 1024]
            loss = loss_fn(net(tr[0][b], tr[1][b], tr[2][b]), tr[3][b], tr[4][b])
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pv = net(*va[:3]).cpu().numpy() * scale[vsi]
        score = wape(sales[vsi, vti][v_ok], pv[v_ok])
        if score < best - 1e-4: best, state, bad = score, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(state); net.eval()
    with torch.no_grad(): pt = net(*te[:3]).cpu().numpy() * scale[tsi]
    return pd.DataFrame({'series_id': pids[tsi], 'dt': dates[tti], 'forecast': pt})

def gru_bt(version):
    out = []
    for s in WEEKS.values():
        runs = [gru_fit_week(version, s, seed) for seed in (0, 1, 2)]          # average 3 seeds
        r = runs[0].copy(); r['forecast'] = np.mean([x.forecast.values for x in runs], 0); out.append(r)
    p = pd.concat(out); p['dt'] = pd.to_datetime(p.dt)
    p = base_bt.merge(p, on=['series_id', 'dt']); p['model'], p['version'] = 'gru', version
    return p

bt_all = pd.concat([bt,
                    cached('bt_lgb_raw', lambda: lgb_bt('sales', 'raw')),
                    cached('bt_lgb_corrected', lambda: lgb_bt('demand', 'corrected')),
                    cached('bt_gru_raw', lambda: gru_bt('raw')),
                    cached('bt_gru_corrected', lambda: gru_bt('corrected')),
                    cached('bt_gru_censored', lambda: gru_bt('censored'))], ignore_index=True)

ft = fair_table(bt_all)
for metric in ['in-stock WAPE (old)', 'observed-hours WAPE', 'censored-aware WAPE', 'observed-hours bias']:
    t = ft.pivot_table(index=['model', 'version'], columns='week', values=metric)[list(WEEKS)]
    t['average'] = t.mean(1)
    print(metric); display(t.round(3))
    if 'WAPE' in metric:
        for m in ['seasonal_naive', 'chronos2', 'lightgbm', 'gru']:
            wins = (t.loc[(m, 'corrected'), list(WEEKS)] < t.loc[(m, 'raw'), list(WEEKS)]).sum()
            print(f"  {m}: corrected beats raw in {wins} of {len(WEEKS)} weeks")
        wins = (t.loc[('gru', 'censored'), list(WEEKS)] < t.loc[('gru', 'corrected'), list(WEEKS)]).sum()
        print(f"  gru: censored beats corrected in {wins} of {len(WEEKS)} weeks")

Loaded bt_lgb_raw from Drive
Loaded bt_lgb_corrected from Drive
Loaded bt_gru_raw from Drive
Loaded bt_gru_corrected from Drive
Loaded bt_gru_censored from Drive
in-stock WAPE (old)


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.366         0.352            0.352   
               raw               0.369         0.365            0.348   
gru            censored          0.392         0.397            0.391   
               corrected         0.398         0.384            0.376   
               raw               0.424         0.405            0.386   
lightgbm       corrected         0.373         0.381            0.375   
               raw               0.396         0.380            0.369   
seasonal_naive corrected         0.450         0.414            0.430   
               raw               0.451         0.417            0.425   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.375              0.316    0.352  
               raw              0.367              0.332    0.356  
gru            censored         0.389              0.331    0.380  
               corrected        0.383              0.351    0.378  
               raw              0.382              0.365    0.392  
lightgbm       corrected        0.386              0.345    0.372  
               raw              0.380              0.386    0.382  
seasonal_naive corrected        0.446              0.384    0.425  
               raw              0.439              0.385    0.424

  seasonal_naive: corrected beats raw in 3 of 5 weeks
  chronos2: corrected beats raw in 3 of 5 weeks
  lightgbm: corrected beats raw in 2 of 5 weeks
  gru: corrected beats raw in 4 of 5 weeks
  gru: censored beats corrected in 2 of 5 weeks
observed-hours WAPE


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.344         0.328            0.335   
               raw               0.385         0.372            0.355   
gru            censored          0.402         0.386            0.383   
               corrected         0.389         0.363            0.358   
               raw               0.457         0.425            0.396   
lightgbm       corrected         0.354         0.345            0.349   
               raw               0.418         0.384            0.370   
seasonal_naive corrected         0.405         0.379            0.397   
               raw               0.436         0.404            0.410   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.361              0.326    0.339  
               raw              0.375              0.359    0.369  
gru            censored         0.384              0.352    0.381  
               corrected        0.373              0.374    0.371  
               raw              0.395              0.396    0.414  
lightgbm       corrected        0.367              0.364    0.356  
               raw              0.388              0.415    0.395  
seasonal_naive corrected        0.416              0.372    0.394  
               raw              0.434              0.393    0.415

  seasonal_naive: corrected beats raw in 5 of 5 weeks
  chronos2: corrected beats raw in 5 of 5 weeks
  lightgbm: corrected beats raw in 5 of 5 weeks
  gru: corrected beats raw in 5 of 5 weeks
  gru: censored beats corrected in 1 of 5 weeks
censored-aware WAPE


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected         0.267         0.253            0.260   
               raw               0.319         0.307            0.291   
gru            censored          0.328         0.310            0.305   
               corrected         0.314         0.288            0.285   
               raw               0.394         0.360            0.330   
lightgbm       corrected         0.276         0.266            0.272   
               raw               0.349         0.317            0.303   
seasonal_naive corrected         0.316         0.295            0.318   
               raw               0.361         0.334            0.345   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected        0.276              0.253    0.262  
               raw              0.302              0.294    0.303  
gru            censored         0.294              0.283    0.304  
               corrected        0.290              0.308    0.297  
               raw              0.319              0.332    0.347  
lightgbm       corrected        0.282              0.295    0.278  
               raw              0.312              0.352    0.327  
seasonal_naive corrected        0.329              0.289    0.310  
               raw              0.359              0.322    0.344

  seasonal_naive: corrected beats raw in 5 of 5 weeks
  chronos2: corrected beats raw in 5 of 5 weeks
  lightgbm: corrected beats raw in 5 of 5 weeks
  gru: corrected beats raw in 5 of 5 weeks
  gru: censored beats corrected in 1 of 5 weeks
observed-hours bias


week                      W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model          version                                                  
chronos2       corrected        -0.027        -0.026            0.027   
               raw              -0.184        -0.174           -0.120   
gru            censored         -0.132        -0.080           -0.030   
               corrected        -0.127        -0.082           -0.033   
               raw              -0.275        -0.236           -0.172   
lightgbm       corrected        -0.054         0.004            0.033   
               raw              -0.228        -0.180           -0.119   
seasonal_naive corrected         0.037         0.010            0.035   
               raw              -0.114        -0.119           -0.090   

week                      W4 Jun 5-11  Test Jun 26-Jul 2  average  
model          version                                             
chronos2       corrected       -0.015             -0.087   -0.026  
               raw             -0.132             -0.200   -0.162  
gru            censored        -0.036             -0.192   -0.094  
               corrected       -0.054             -0.227   -0.105  
               raw             -0.155             -0.269   -0.221  
lightgbm       corrected       -0.003             -0.173   -0.039  
               raw             -0.140             -0.292   -0.192  
seasonal_naive corrected       -0.017             -0.006    0.012  
               raw             -0.131             -0.126   -0.116

## Step 13: Hand-off to Stream 5 (all 5 weeks, with forecast ranges)
Stream 5 orders a forecast percentile each day, so it needs ranges, not just point forecasts. Chronos-2 outputs 10th/50th/90th percentiles; for LightGBM we train stream 2's quantile models per week. We check calibration (share of in-stock days with sales inside the 10th to 90th range, target 80%) and save everything in one file.

In [25]:
Q_PARAMS = dict(n_estimators=800, learning_rate=0.05, num_leaves=63, min_child_samples=100, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def lgb_q_bt(value_col, version):
    """Stream 2's quantile LightGBM (10th/50th/90th), retrained per backtest week like Step 12c."""
    out = []
    for s in WEEKS.values():
        s = pd.Timestamp(s); cutoff = s - pd.Timedelta(days=7)
        d = lgb_features(value_col, cutoff)
        tr = d[(d.dt <= cutoff - pd.Timedelta(days=7)) & d.lag7.notna()]
        va = d[(d.dt > cutoff - pd.Timedelta(days=7)) & (d.dt <= cutoff)]
        te = d[(d.dt >= s) & (d.dt < s + pd.Timedelta(days=7))]
        r = te[['series_id', 'dt']].copy()
        for q in [0.1, 0.5, 0.9]:
            m = lgb.LGBMRegressor(objective='quantile', alpha=q, **Q_PARAMS).fit(
                tr[LGB_FEATS], tr.target, eval_set=[(va[LGB_FEATS], va.target)],
                categorical_feature=['first_category_id'], callbacks=[lgb.early_stopping(50, verbose=False)])
            r[f'q{int(q * 100)}'] = m.predict(te[LGB_FEATS])
        r[['q10', 'q50', 'q90']] = np.sort(r[['q10', 'q50', 'q90']].to_numpy(), axis=1).clip(min=0)   # keep low <= mid <= high
        out.append(r)
    p = pd.concat(out); p['model'], p['version'] = 'lightgbm', version
    return p

lgb_q = pd.concat([cached('bt_lgb_q_raw', lambda: lgb_q_bt('sales', 'raw')),
                   cached('bt_lgb_q_corrected', lambda: lgb_q_bt('demand', 'corrected'))])

# Attach the LightGBM ranges to its backtest rows; Chronos-2 already has them
hand = bt_all.drop(columns=['q10', 'q50', 'q90'], errors='ignore').merge(
    pd.concat([lgb_q, bt_all[bt_all.model == 'chronos2'][['series_id', 'dt', 'model', 'version', 'q10', 'q50', 'q90']]]),
    on=['series_id', 'dt', 'model', 'version'], how='left')

# Calibration of the ranges on in-stock days, per model and week (target: 80% inside 10th-90th)
s = hand[(hand.oos_hours == 0) & hand.q10.notna()]
cov = (s.assign(inside=(s.sales >= s.q10) & (s.sales <= s.q90))
         .pivot_table(index=['model', 'version'], columns='week', values='inside')[list(WEEKS)])
cov['average'] = cov.mean(1)
print("Share of in-stock days inside the 10th-90th range (target 0.80):"); display(cov.round(3))

# Save one file for Stream 5
cols = ['series_id', 'dt', 'week', 'sales', 'demand', 'oos_hours', 'stockout_band', 'model', 'version',
        'forecast', 'q10', 'q50', 'q90']
hand[cols].to_parquet(PROC_DIR / 'forecasts_stream4_backtest_v1.parquet', index=False)
print("Saved forecasts_stream4_backtest_v1.parquet:", hand[cols].shape)
print(hand.groupby(['model', 'version']).size())

Loaded bt_lgb_q_raw from Drive
Loaded bt_lgb_q_corrected from Drive
Share of in-stock days inside the 10th-90th range (target 0.80):


week                W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  W4 Jun 5-11  \
model    version                                                               
chronos2 corrected         0.821         0.816            0.795        0.789   
         raw               0.856         0.843            0.836        0.818   
lightgbm corrected         0.696         0.663            0.684        0.685   
         raw               0.762         0.754            0.745        0.750   

week                Test Jun 26-Jul 2  average  
model    version                                
chronos2 corrected              0.787    0.802  
         raw                    0.795    0.830  
lightgbm corrected              0.759    0.698  
         raw                    0.721    0.746

Saved forecasts_stream4_backtest_v1.parquet: (630000, 13)
model           version  
chronos2        corrected    70000
                raw          70000
gru             censored     70000
                corrected    70000
                raw          70000
lightgbm        corrected    70000
                raw          70000
seasonal_naive  corrected    70000
                raw          70000
dtype: int64


## Step 13b: TFT on the fair scores (test week)
John's TFT was only scored on in-stock days. We rescore his test-week forecasts with the observed-hours and censored-aware scores, next to Chronos-2, LightGBM and seasonal naive. TFT was not retrained for the backtest weeks (too costly), so this covers the test week only.

In [27]:
# John's TFT forecasts + stream 2 + Chronos-2, test week only
tft = pd.read_parquet(PROC_DIR / 'forecasts_stream2_tft_v1.parquet')
s4 = pd.read_parquet(PROC_DIR / 'forecasts_stream4_v1.parquet')
allf = pd.concat([fc2, tft, s4], ignore_index=True)
allf['dt'] = pd.to_datetime(allf['dt'])
allf = allf[allf.set == 'test']

# Hourly profile per product from in-stock hours up to May 7
h = pd.read_parquet(PROC_DIR / 'hourly_demand_v1.parquet',
                    columns=['series_id', 'dt', 'hour', 'sales', 'out_of_stock'])
h['dt'] = pd.to_datetime(h['dt'])
prof = (h[(h.dt <= '2024-05-07') & (h.out_of_stock == 0)]
        .groupby(['series_id', 'hour']).sales.mean().rename('m').reset_index())
prof['share'] = prof.m / prof.groupby('series_id').m.transform('sum')

# Each test day: sales in its in-stock hours, and the share of the day those hours cover
ht = (h[h.dt.isin(allf.dt.unique()) & (h.out_of_stock == 0)]
      .merge(prof[['series_id', 'hour', 'share']], on=['series_id', 'hour'], how='left'))
ht['share'] = ht.share.fillna(0)
obs = ht.groupby(['series_id', 'dt']).agg(obs_sales=('sales', 'sum'),
                                          obs_share=('share', 'sum')).reset_index()
f = allf.merge(obs, on=['series_id', 'dt'], how='left').fillna({'obs_sales': 0, 'obs_share': 0})

rows = []
for (m, v), g in f.groupby(['model', 'version']):
    ins = g[g.oos_hours == 0]
    fo = g.forecast * g.obs_share
    cens = np.where(g.oos_hours == 0, (g.sales - g.forecast).abs(),
                    np.clip(g.sales - g.forecast, 0, None))
    rows.append({'model': m, 'version': v,
                 'instock_wape': wape(ins.sales, ins.forecast),
                 'obs_wape': (g.obs_sales - fo).abs().sum() / g.obs_sales.sum(),
                 'obs_bias': (fo - g.obs_sales).sum() / g.obs_sales.sum(),
                 'cens_wape': cens.sum() / g.sales.sum()})
tft_fair = pd.DataFrame(rows).round(3).sort_values('obs_wape')
tft_fair

,model,version,instock_wape,obs_wape,obs_bias,cens_wape
0,chronos2,corrected,0.316,0.328,-0.089,0.253
1,chronos2,raw,0.332,0.361,-0.201,0.294
6,tft,corrected,0.344,0.368,-0.203,0.298
2,lightgbm,corrected,0.350,0.370,-0.171,0.298
4,seasonal_naive,corrected,0.384,0.373,-0.008,0.289
5,seasonal_naive,raw,0.385,0.395,-0.128,0.322
7,tft,raw,0.377,0.406,-0.262,0.339
3,lightgbm,raw,0.389,0.419,-0.292,0.354


## Step 14: Censored network on all 50,000 products
Corrected demand exists only for our 2,000 products, because the hourly fill is expensive. The censored network needs only raw sales and stockout hours, which exist for all 50,000 products, so it can learn from 25 times more data. We train it per backtest week on all 50,000 (raw sales of the selling hours 6:00 to 22:00, matching our daily sales), score it on the same 2,000 products with the fair scores, and compare it with the corrected network trained on 2,000 products. One seed per version to keep training time reasonable.

In [26]:
import pyarrow.parquet as pq
from src.load_data import RAW_DIR, _series_id

# ---- 1. Selling-hours sales (6:00-22:00) for all 50,000 products, same definition as our daily `sales`
parts = []
for split in ['train', 'eval']:
    pf = pq.ParquetFile(RAW_DIR / 'data' / f'{split}.parquet')
    for i in range(pf.num_row_groups):
        t = pf.read_row_group(i, columns=['store_id', 'product_id', 'dt', 'hours_sale'])
        hs = t.column('hours_sale').combine_chunks().flatten().to_numpy().reshape(-1, 24)
        parts.append(pd.DataFrame({'series_id': _series_id(t.column('store_id').to_numpy(), t.column('product_id').to_numpy()),
                                   'dt': pd.to_datetime(np.asarray(t.column('dt').to_numpy(zero_copy_only=False))),
                                   'sales_sell': hs[:, 6:22].sum(1).astype('float32')}))
full = pd.concat(parts, ignore_index=True); del parts

# ---- 2. Products x days grids for all 50,000 products
bcols = ['series_id', 'dt', 'stock_hour6_22_cnt', 'discount', 'discount_missing', 'holiday_flag', 'activity_flag',
         'precpt', 'avg_temperature', 'first_category_id', 'day_of_week']
big = pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=bcols); big['dt'] = pd.to_datetime(big['dt'])
big = big.merge(full, on=['series_id', 'dt']).sort_values(['series_id', 'dt']).reset_index(drop=True); del full
big['discount'] = big.discount.fillna(big.groupby('series_id').discount.transform('median')).fillna(1.0)
BIDS = big.series_id.unique(); BS, BT = len(BIDS), big.dt.nunique(); assert len(big) == BS * BT
bg = lambda c: big[c].to_numpy('float32').reshape(BS, BT)
b_sales, b_oos = bg('sales_sell'), bg('stock_hour6_22_cnt')
b_dow = big.day_of_week.to_numpy().reshape(BS, BT)
b_X = np.stack([bg('discount'), bg('discount_missing'), bg('holiday_flag'), bg('activity_flag'),
                bg('precpt') / 10, (bg('avg_temperature') - 22) / 4], -1)
b_cat = pd.factorize(big.groupby('series_id', sort=False).first_category_id.first())[0]
b_dates = np.sort(big.dt.unique()); b_day = {t: i for i, t in enumerate(pd.to_datetime(b_dates))}
b_sample = np.where(pd.Index(BIDS).isin(sample_ids))[0]
del big
print(f"{BS:,} products x {BT} days | {len(b_sample):,} in our sample")

# ---- 3. Same GRU and censored loss as Steps 11-12c, trained on a chosen set of products, predicting our 2,000
def gru_big_week(version, s, seed, prod_idx, batch=4096):
    torch.manual_seed(seed); np.random.seed(seed)
    t0 = b_day[pd.Timestamp(s)]; cut = t0 - 7
    tr_ts, va_ts, te_ts = np.arange(7 + W_BT - 1, cut - 6), np.arange(cut - 6, cut + 1), np.arange(t0, t0 + 7)
    known = np.arange(0, cut + 1)
    if version == 'censored':
        mm = np.where(b_oos[:, known] == 0, b_sales[:, known], np.nan); sc = np.nanmean(mm, 1)
        scale = np.where(np.isnan(sc), b_sales[:, known].mean(1), sc)
    else:
        scale = b_sales[:, known].mean(1)
    scale = np.maximum(scale, 0.01).astype('float32')
    def rows(prods, ts):
        si, ti = [a.ravel() for a in np.meshgrid(prods, ts, indexing='ij')]
        win = ti[:, None] - 7 - W_BT + 1 + np.arange(W_BT)
        feats = [b_sales[si[:, None], win] / scale[si, None]]
        if version == 'censored': feats.append(b_oos[si[:, None], win] / 16)
        xs = np.stack(feats, -1).astype('float32')
        xd = np.concatenate([b_X[si, ti], np.eye(7, dtype='float32')[b_dow[si, ti]]], 1)
        y = (b_sales[si, ti] / scale[si]).astype('float32')
        cens = ((b_oos[si, ti] > 0) & (version == 'censored')).astype('float32')
        return [torch.tensor(a).to(dev) for a in (xs, xd, b_cat[si], y, cens)], si, ti
    (tr, _, _), (va, vsi, vti) = rows(prod_idx, tr_ts), rows(prod_idx, va_ts)
    te, tsi, tti = rows(b_sample, te_ts)
    net = Net(tr[0].shape[-1], b_cat.max() + 1).to(dev); opt = torch.optim.Adam(net.parameters(), 2e-3)
    v_ok = b_oos[vsi, vti] == 0; best, state, bad = 1e9, None, 0
    for ep in range(40):
        net.train(); perm = torch.randperm(len(tr[3]), device=dev)
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]
            loss = loss_fn(net(tr[0][b], tr[1][b], tr[2][b]), tr[3][b], tr[4][b])
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pv = torch.cat([net(va[0][i:i + 65536], va[1][i:i + 65536], va[2][i:i + 65536])
                            for i in range(0, len(va[3]), 65536)]).cpu().numpy() * scale[vsi]
        score = wape(b_sales[vsi, vti][v_ok], pv[v_ok])                       # early stopping on in-stock valid days
        if score < best - 1e-4: best, state, bad = score, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(state); net.eval()
    with torch.no_grad(): pt = net(*te[:3]).cpu().numpy() * scale[tsi]
    print(f"    {version} {s}: {len(tr[3]):,} training rows, best valid {best:.3f}, {ep + 1} epochs", flush=True)
    return pd.DataFrame({'series_id': BIDS[tsi], 'dt': pd.to_datetime(b_dates[tti]), 'forecast': pt})

runs = {'censored, 50k': ('censored', np.arange(BS)),
        'raw, 50k':      ('raw', np.arange(BS)),
        'censored, 2k':  ('censored', b_sample)}               # same settings on our 2,000, for a clean comparison
new = []
for name, (version, prods) in runs.items():
    weeks = [cached(f"bt50k_{name.replace(', ', '_')}_{i}", lambda s=s: gru_big_week(version, s, 0, prods))
             for i, s in enumerate(WEEKS.values())]
    p = base_bt.merge(pd.concat(weeks), on=['series_id', 'dt']); p['model'], p['version'] = 'gru', name
    new.append(p)

# ---- 4. Fair scores: new runs vs. the 2,000-product GRUs (Step 12c) and the best corrected models
ref = bt_all[(bt_all.model.isin(['gru', 'chronos2', 'lightgbm'])) & bt_all.version.isin(['corrected', 'censored', 'raw'])]
ft14 = fair_table(pd.concat([ref] + new, ignore_index=True))
for metric in ['observed-hours WAPE', 'censored-aware WAPE', 'observed-hours bias']:
    t = ft14.pivot_table(index=['model', 'version'], columns='week', values=metric)[list(WEEKS)]
    t['average'] = t.mean(1)
    print(metric); display(t.round(3))
    if 'WAPE' in metric:
        for a, b in [(('gru', 'censored, 50k'), ('gru', 'corrected')), (('gru', 'censored, 50k'), ('gru', 'censored, 2k')),
                     (('gru', 'censored, 50k'), ('gru', 'raw, 50k')), (('gru', 'censored, 50k'), ('lightgbm', 'corrected'))]:
            wins = (t.loc[a, list(WEEKS)] < t.loc[b, list(WEEKS)]).sum()
            print(f"  {a[1]} GRU beats {b[0]} {b[1]} in {wins} of {len(WEEKS)} weeks")

50,000 products x 97 days | 2,000 in our sample
    censored 2024-05-15: 750,000 training rows, best valid 0.362, 7 epochs
Saved bt50k_censored_50k_0 to Drive
    censored 2024-05-22: 1,100,000 training rows, best valid 0.349, 10 epochs
Saved bt50k_censored_50k_1 to Drive
    censored 2024-05-29: 1,450,000 training rows, best valid 0.336, 7 epochs
Saved bt50k_censored_50k_2 to Drive
    censored 2024-06-05: 1,800,000 training rows, best valid 0.320, 13 epochs
Saved bt50k_censored_50k_3 to Drive
    censored 2024-06-26: 2,850,000 training rows, best valid 0.327, 12 epochs
Saved bt50k_censored_50k_4 to Drive
    raw 2024-05-15: 750,000 training rows, best valid 0.367, 9 epochs
Saved bt50k_raw_50k_0 to Drive
    raw 2024-05-22: 1,100,000 training rows, best valid 0.351, 21 epochs
Saved bt50k_raw_50k_1 to Drive
    raw 2024-05-29: 1,450,000 training rows, best valid 0.345, 13 epochs
Saved bt50k_raw_50k_2 to Drive
    raw 2024-06-05: 1,800,000 training rows, best valid 0.329, 10 epochs
Save

week                    W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model    version                                                      
chronos2 corrected             0.344         0.328            0.335   
         raw                   0.385         0.372            0.355   
gru      censored              0.402         0.386            0.383   
         censored, 2k          0.431         0.390            0.389   
         censored, 50k         0.407         0.382            0.378   
         corrected             0.389         0.363            0.358   
         raw                   0.457         0.425            0.396   
         raw, 50k              0.493         0.440            0.418   
lightgbm corrected             0.354         0.345            0.349   
         raw                   0.418         0.384            0.370   

week                    W4 Jun 5-11  Test Jun 26-Jul 2  average  
model    version                                                 
chronos2 corrected            0.361              0.326    0.339  
         raw                  0.375              0.359    0.369  
gru      censored             0.384              0.352    0.381  
         censored, 2k         0.397              0.349    0.391  
         censored, 50k        0.391              0.367    0.385  
         corrected            0.373              0.374    0.371  
         raw                  0.395              0.396    0.414  
         raw, 50k             0.423              0.444    0.444  
lightgbm corrected            0.367              0.364    0.356  
         raw                  0.388              0.415    0.395

  censored, 50k GRU beats gru corrected in 1 of 5 weeks
  censored, 50k GRU beats gru censored, 2k in 4 of 5 weeks
  censored, 50k GRU beats gru raw, 50k in 5 of 5 weeks
  censored, 50k GRU beats lightgbm corrected in 0 of 5 weeks
censored-aware WAPE


week                    W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model    version                                                      
chronos2 corrected             0.267         0.253            0.260   
         raw                   0.319         0.307            0.291   
gru      censored              0.328         0.310            0.305   
         censored, 2k          0.363         0.311            0.309   
         censored, 50k         0.318         0.293            0.301   
         corrected             0.314         0.288            0.285   
         raw                   0.394         0.360            0.330   
         raw, 50k              0.432         0.375            0.353   
lightgbm corrected             0.276         0.266            0.272   
         raw                   0.349         0.317            0.303   

week                    W4 Jun 5-11  Test Jun 26-Jul 2  average  
model    version                                                 
chronos2 corrected            0.276              0.253    0.262  
         raw                  0.302              0.294    0.303  
gru      censored             0.294              0.283    0.304  
         censored, 2k         0.303              0.277    0.313  
         censored, 50k        0.293              0.301    0.301  
         corrected            0.290              0.308    0.297  
         raw                  0.319              0.332    0.347  
         raw, 50k             0.350              0.384    0.379  
lightgbm corrected            0.282              0.295    0.278  
         raw                  0.312              0.352    0.327

  censored, 50k GRU beats gru corrected in 1 of 5 weeks
  censored, 50k GRU beats gru censored, 2k in 4 of 5 weeks
  censored, 50k GRU beats gru raw, 50k in 5 of 5 weeks
  censored, 50k GRU beats lightgbm corrected in 0 of 5 weeks
observed-hours bias


week                    W1 May 15-21  W2 May 22-28  W3 May 29-Jun 4  \
model    version                                                      
chronos2 corrected            -0.027        -0.026            0.027   
         raw                  -0.184        -0.174           -0.120   
gru      censored             -0.132        -0.080           -0.030   
         censored, 2k         -0.222        -0.056           -0.015   
         censored, 50k        -0.112        -0.010           -0.048   
         corrected            -0.127        -0.082           -0.033   
         raw                  -0.275        -0.236           -0.172   
         raw, 50k             -0.322        -0.213           -0.189   
lightgbm corrected            -0.054         0.004            0.033   
         raw                  -0.228        -0.180           -0.119   

week                    W4 Jun 5-11  Test Jun 26-Jul 2  average  
model    version                                                 
chronos2 corrected           -0.015             -0.087   -0.026  
         raw                 -0.132             -0.200   -0.162  
gru      censored            -0.036             -0.192   -0.094  
         censored, 2k        -0.009             -0.153   -0.091  
         censored, 50k       -0.017             -0.233   -0.084  
         corrected           -0.054             -0.227   -0.105  
         raw                 -0.155             -0.269   -0.221  
         raw, 50k            -0.209             -0.354   -0.258  
lightgbm corrected           -0.003             -0.173   -0.039  
         raw                 -0.140             -0.292   -0.192

## Step 15: Error range, sign test and the quiet test week
On stockout days true demand is unknown, so each score gets a range: a lower bound (only forecasts below recorded sales count as error) and an upper bound (demand = sales scaled up by the share of the day that was in stock). A sign test checks whether 5 of 5 weekly wins could be luck. The last cell explains why the test week looked different.

In [28]:
from scipy.stats import binomtest

bt = pd.read_parquet(PROC_DIR / 'forecasts_stream4_backtest_v1.parquet')
bt['dt'] = pd.to_datetime(bt['dt'])
week_start = {'W1': '2024-05-15', 'W2': '2024-05-22', 'W3': '2024-05-29', 'W4': '2024-06-05', 'Test': '2024-06-26'}
bt['wk'] = None
for w, a in week_start.items():
    a = pd.Timestamp(a)
    bt.loc[(bt.dt >= a) & (bt.dt <= a + pd.Timedelta(days=6)), 'wk'] = w
bt = bt[bt.wk.notna()]

# Hourly profile from in-stock hours up to May 7, then each day's in-stock sales and share of the day
h = pd.read_parquet(PROC_DIR / 'hourly_demand_v1.parquet', columns=['series_id', 'dt', 'hour', 'sales', 'out_of_stock'])
h['dt'] = pd.to_datetime(h['dt'])
prof = h[(h.dt <= '2024-05-07') & (h.out_of_stock == 0)].groupby(['series_id', 'hour']).sales.mean().rename('m').reset_index()
prof['share'] = prof.m / prof.groupby('series_id').m.transform('sum')
ht = h[h.dt.isin(bt.dt.unique()) & (h.out_of_stock == 0)].merge(prof[['series_id', 'hour', 'share']], on=['series_id', 'hour'], how='left')
ht['share'] = ht.share.fillna(0)
obs = ht.groupby(['series_id', 'dt']).agg(obs_sales=('sales', 'sum'), obs_share=('share', 'sum')).reset_index()
f = bt.merge(obs, on=['series_id', 'dt'], how='left').fillna({'obs_sales': 0, 'obs_share': 0})

# Lower bound: demand >= sales on stockout days. Upper bound: sales scaled up by the in-stock share of the day
so = f.oos_hours > 0
f['lo_err'] = np.where(so, np.clip(f.sales - f.forecast, 0, None), (f.sales - f.forecast).abs())
f['ub_truth'] = np.where(so, np.maximum(f.sales, f.obs_sales / np.maximum(f.obs_share, 0.3)), f.sales)
f['hi_err'] = (f.ub_truth - f.forecast).abs()
g = f.groupby(['model', 'version', 'wk'])
rng = pd.DataFrame({'lower': g.lo_err.sum() / g.sales.sum(), 'upper': g.hi_err.sum() / g.ub_truth.sum()}).reset_index()

# 5-week average, weeks where corrected beats raw, and one-sided sign test
summary = []
for m, gm in rng.groupby('model'):
    p = gm.pivot(index='wk', columns='version', values=['lower', 'upper'])
    row = {'model': m}
    for b in ['lower', 'upper']:
        wins = int((p[(b, 'corrected')] < p[(b, 'raw')]).sum()); n = len(p)
        row[f'{b} raw'] = p[(b, 'raw')].mean(); row[f'{b} corrected'] = p[(b, 'corrected')].mean()
        row[f'{b} wins'] = f'{wins}/{n}'
        row[f'{b} p'] = binomtest(wins, n, 0.5, alternative='greater').pvalue
    summary.append(row)
pd.DataFrame(summary).round(3)

,model,lower raw,lower corrected,lower wins,lower p,upper raw,upper corrected,upper wins,upper p
0,chronos2,0.303,0.262,5/5,0.031,0.411,0.383,5/5,0.031
1,gru,0.347,0.297,5/5,0.031,0.452,0.412,5/5,0.031
2,lightgbm,0.327,0.278,5/5,0.031,0.434,0.397,5/5,0.031
3,seasonal_naive,0.344,0.310,5/5,0.031,0.452,0.432,5/5,0.031


In [29]:
rows = []
for w, a in week_start.items():
    a = pd.Timestamp(a); b = a + pd.Timedelta(days=6)
    hist = dd[(dd.dt < a - pd.Timedelta(days=6)) & (dd.dt >= a - pd.Timedelta(days=34))].groupby('series_id').demand.mean().rename('hist_avg')
    g = dd[(dd.dt >= a) & (dd.dt <= b)].merge(hist, on='series_id')
    ins = g.oos_hours == 0
    week_avg = g.groupby('series_id').demand.transform('mean')
    rows.append({'week': w,
                 'stockout days': (~ins).mean(),
                 'in-stock days vs. same week': g.sales[ins].sum() / week_avg[ins].sum() - 1,
                 'in-stock days vs. prior 4 weeks': g.sales[ins].sum() / g.hist_avg[ins].sum() - 1,
                 'week demand vs. prior 4 weeks': g.demand.sum() / g.hist_avg.sum() - 1})
pd.DataFrame(rows).round(3)

,week,stockout days,in-stock days vs. same week,in-stock days vs. prior 4 weeks,week demand vs. prior 4 weeks
0,W1,0.398,-0.090,-0.136,0.011
1,W2,0.399,-0.076,-0.110,0.008
2,W3,0.370,-0.078,-0.126,-0.014
3,W4,0.408,-0.093,-0.133,0.007
4,Test,0.408,-0.071,0.019,0.131


## Step 16: Semi-synthetic test (known true demand)
On real data, true demand on stockout days is unknown, so we can't prove which score is right. Here we build a world where we know it: take corrected demand as the "truth", give each product-day a simulated stock level (60th percentile of the last 28 days, with noise), and stop sales once stock runs out. That gives about 41% stockout days, landing on busy days like the real data. Then we run the full pipeline (v1 filler, then Chronos-2) and check whether each score ranks the forecasts the same way the true error does.

In [30]:
SYN = PROC_DIR / 'semisyn'; SYN.mkdir(exist_ok=True)
h = pd.read_parquet(PROC_DIR / 'hourly_demand_v1.parquet', columns=['series_id', 'dt', 'hour', 'split', 'demand'])
h['dt'] = pd.to_datetime(h['dt']); h = h.sort_values(['series_id', 'dt', 'hour']).reset_index(drop=True)
S, D = h.series_id.nunique(), h.dt.nunique()
true = h.demand.to_numpy('float64').reshape(S, D, 16)            # "true" hourly demand in this world
ref = pd.DataFrame(true.sum(2).T).shift(1).rolling(28, min_periods=7).quantile(0.6).to_numpy().T  # past 28 days only
noise = np.exp(np.random.default_rng(0).normal(0, 0.25, (S, D)))
cum = true.cumsum(2); before = cum - true

def censor(c):
    inv = np.where(np.isnan(ref), np.inf, c * ref * noise)[:, :, None]   # simulated stock for the day
    sales = np.clip(inv - before, 0, true)                                # sales stop once stock runs out
    oos = np.maximum.accumulate((cum >= inv - 1e-9) & (cum > 0), axis=2)  # empty for the rest of the day
    return sales, oos

lo, hi = 0.5, 3.0                      # tune stock level so ~41% of days stock out (real data: 44%)
for _ in range(25):
    c = (lo + hi) / 2; s, o = censor(c)
    lo, hi = (c, hi) if o.any(2)[:, 7:].mean() > 0.41 else (lo, c)
s, o = censor(c)
syn = pd.DataFrame({'series_id': h.series_id, 'dt': h.dt, 'hour': h.hour, 'split': h.split,
                    'sales': s.reshape(-1), 'out_of_stock': o.reshape(-1).astype('int8'), 'true': true.reshape(-1)})
syn.to_parquet(SYN / 'hourly_syn.parquet', index=False)
dsyn = syn.groupby(['series_id', 'dt']).agg(sales=('sales', 'sum'), true=('true', 'sum'), oos_hours=('out_of_stock', 'sum')).reset_index()
ins = dsyn.oos_hours == 0
print(f'stockout days {(~ins).mean():.1%} | hidden demand {1 - s.sum() / true.sum():.1%} | '
      f'true demand on stockout days {dsyn.true[~ins].sum() / dsyn.sales[~ins].sum() - 1:.1%} above recorded | '
      f'in-stock days {dsyn.true[ins].sum() / dsyn.groupby("series_id").true.transform("mean")[ins].sum() - 1:.1%} vs product average')

stockout days 38.0% | hidden demand 15.0% | true demand on stockout days 41.6% above recorded | in-stock days -19.6% vs product average


In [33]:
import json, glob
from pathlib import Path
nb_path = glob.glob('/content/*/notebooks/missed_demand.ipynb')[0]
nb = json.load(open(nb_path))
code = '\n'.join(''.join(c['source']) for c in nb['cells'][3:] if c['cell_type'] == 'code')
code = (code.replace("hourly = load_hourly(sample_ids)", f"hourly = pd.read_parquet('{SYN / 'hourly_syn.parquet'}').drop(columns=['true'])")
            .replace("PROC_DIR / 'hourly_demand_v1.parquet'", f"Path('{SYN / 'hourly_demand_syn.parquet'}')")
            .replace("PROC_DIR / 'daily_demand_v1.parquet'", f"Path('{SYN / 'daily_demand_syn.parquet'}')"))
assert code.count(str(SYN)) == 3, 'notebook changed, tell Claude'
exec('from pathlib import Path\nfrom IPython.display import display\n' + code, {})

3,104,000 hourly rows for 2,000 products
Share of selling hours out of stock: 0.157
Test days: 11,220 | hidden hours: 39,919 | avg block 3.6 h
Practice hours used for training: 144,589


,wape,bias
Raw sales (assume zero),1.000,-1.000
Historical average,0.517,0.072
Day-adjusted average,0.512,-0.011
LightGBM,0.477,-0.018


epoch 1: training loss 1.9518
epoch 2: training loss 1.8785
epoch 3: training loss 1.9225
epoch 4: training loss 1.9000
epoch 5: training loss 1.8825
epoch 6: training loss 1.9101
epoch 7: training loss 1.8695
epoch 8: training loss 1.8813
Missed demand = 10.7% of recorded sales (selling hours, 2,000 products)
On stockout days, true demand is 25.2% above recorded sales
Share of missed demand on days that still show sales (invisible in daily data): 100.0%
Saved hourly_demand_v1.parquet (3,104,000 rows) and daily_demand_v1.parquet (194,000 rows)


In [34]:
from chronos import Chronos2Pipeline
pipe_syn = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
ds = pd.read_parquet(SYN / 'daily_demand_syn.parquet')[['series_id', 'dt', 'sales', 'demand', 'oos_hours']]
ds['dt'] = pd.to_datetime(ds['dt'])
ds = ds.merge(dsyn[['series_id', 'dt', 'true']], on=['series_id', 'dt'])
week_start = {'W1': '2024-05-15', 'W2': '2024-05-22', 'W3': '2024-05-29', 'W4': '2024-06-05', 'Test': '2024-06-26'}
syn_days = [pd.Timestamp(a) + pd.Timedelta(days=i) for a in week_start.values() for i in range(7)]

def make_syn_forecasts():
    out = []
    for col, ver in [('sales', 'raw'), ('demand', 'corrected'), ('true', 'oracle')]:   # oracle = perfect correction
        for day in syn_days:
            ctx = ds[ds.dt <= day - pd.Timedelta(days=7)][['series_id', 'dt', col]].rename(
                columns={'series_id': 'id', 'dt': 'timestamp', col: 'target'})
            p = pipe_syn.predict_df(ctx, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9], id_column='id',
                                    timestamp_column='timestamp', target='target', batch_size=256)
            p = p[p.timestamp == day].rename(columns={'id': 'series_id', 'timestamp': 'dt', 'predictions': 'forecast'})
            out.append(p[['series_id', 'dt', 'forecast']].assign(model='chronos2', version=ver))
            sn = ds[ds.dt == day - pd.Timedelta(days=7)][['series_id', col]].rename(columns={col: 'forecast'})
            out.append(sn.assign(dt=day, model='seasonal_naive', version=ver))
    f = pd.concat(out, ignore_index=True); f['forecast'] = f.forecast.clip(lower=0)
    return f
fsyn = cached('semisyn_forecasts', make_syn_forecasts)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Saved semisyn_forecasts to Drive


In [35]:
from scipy.stats import spearmanr
prof = syn[(syn.dt <= '2024-05-07') & (syn.out_of_stock == 0)].groupby(['series_id', 'hour']).sales.mean().rename('m').reset_index()
prof['share'] = prof.m / prof.groupby('series_id').m.transform('sum')
ht = syn[syn.out_of_stock == 0].merge(prof[['series_id', 'hour', 'share']], on=['series_id', 'hour'], how='left').fillna({'share': 0})
obs = ht.groupby(['series_id', 'dt']).agg(obs_sales=('sales', 'sum'), obs_share=('share', 'sum')).reset_index()
f = (fsyn.assign(dt=pd.to_datetime(fsyn.dt)).merge(dsyn, on=['series_id', 'dt'])
         .merge(obs, on=['series_id', 'dt'], how='left').fillna({'obs_sales': 0, 'obs_share': 0}))
f['wk'] = None
for w, a in week_start.items():
    a = pd.Timestamp(a); f.loc[(f.dt >= a) & (f.dt <= a + pd.Timedelta(days=6)), 'wk'] = w
so = f.oos_hours > 0
f['lo'] = np.where(so, np.clip(f.sales - f.forecast, 0, None), (f.sales - f.forecast).abs())
f['ub'] = np.where(so, np.maximum(f.sales, f.obs_sales / np.maximum(f.obs_share, 0.3)), f.sales)
rows = []
for (m, v, w), g in f.groupby(['model', 'version', 'wk']):
    i = g[g.oos_hours == 0]
    rows.append({'model': m, 'version': v, 'wk': w,
                 'TRUE wape': (g.true - g.forecast).abs().sum() / g.true.sum(),
                 'TRUE bias': (g.forecast - g.true).sum() / g.true.sum(),
                 'in-stock': (i.sales - i.forecast).abs().sum() / i.sales.sum(),
                 'observed-hours': (g.obs_sales - g.forecast * g.obs_share).abs().sum() / g.obs_sales.sum(),
                 'lower': g.lo.sum() / g.sales.sum(),
                 'upper': (g.ub - g.forecast).abs().sum() / g.ub.sum()})
syn_scores = pd.DataFrame(rows)
display(syn_scores.groupby(['model', 'version']).mean(numeric_only=True).round(3).sort_values('TRUE wape'))

scores = ['TRUE wape', 'in-stock', 'observed-hours', 'lower', 'upper']
print('Weeks (of 5) where corrected beats raw, by score:')
for m, gm in syn_scores[syn_scores.version.isin(['raw', 'corrected'])].groupby('model'):
    p = gm.pivot(index='wk', columns='version')
    print(' ', m, {s: int((p[(s, 'corrected')] < p[(s, 'raw')]).sum()) for s in scores})
print('Rank agreement with the TRUE score (1 = same ranking of all 6 forecasts):')
print(' ', {s: round(np.mean([spearmanr(g['TRUE wape'], g[s])[0] for _, g in syn_scores.groupby('wk')]), 2) for s in scores[1:]})

TRUE wape  TRUE bias  in-stock  observed-hours  \
model          version                                                     
chronos2       corrected      0.308     -0.066     0.309           0.301   
               oracle         0.308     -0.034     0.328           0.308   
               raw            0.336     -0.163     0.289           0.313   
seasonal_naive corrected      0.343     -0.061     0.376           0.349   
               raw            0.357     -0.153     0.343           0.347   
               oracle         0.366      0.003     0.427           0.385   

                          lower  upper  
model          version                  
chronos2       corrected  0.219  0.314  
               oracle     0.224  0.316  
               raw        0.235  0.337  
seasonal_naive corrected  0.265  0.352  
               raw        0.268  0.362  
               oracle     0.290  0.382

Weeks (of 5) where corrected beats raw, by score:
  chronos2 {'TRUE wape': 5, 'in-stock': 0, 'observed-hours': 4, 'lower': 5, 'upper': 5}
  seasonal_naive {'TRUE wape': 5, 'in-stock': 0, 'observed-hours': 1, 'lower': 3, 'upper': 5}
Rank agreement with the TRUE score (1 = same ranking of all 6 forecasts):
  {'in-stock': np.float64(0.58), 'observed-hours': np.float64(0.79), 'lower': np.float64(0.9), 'upper': np.float64(0.93)}


## Step 17: Finer forecast ranges for Stream 5
The simulator orders at the quantile that matches the margin (20% margin → q20, 30% → q30, 40% → q40). The backtest file only has q10/q50/q90, so we rerun Chronos-2 for all 5 weeks with quantiles from 0.05 to 0.95.

In [36]:
from chronos import Chronos2Pipeline
pipe_q = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
QS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
week_start = {'W1': '2024-05-15', 'W2': '2024-05-22', 'W3': '2024-05-29', 'W4': '2024-06-05', 'Test': '2024-06-26'}

def make_quantile_forecasts():
    out = []
    for w, a in week_start.items():
        for i in range(7):
            day = pd.Timestamp(a) + pd.Timedelta(days=i)
            for col, ver in [('sales', 'raw'), ('demand', 'corrected')]:
                ctx = dd[dd.dt <= day - pd.Timedelta(days=7)][['series_id', 'dt', col]].rename(
                    columns={'series_id': 'id', 'dt': 'timestamp', col: 'target'})
                p = pipe_q.predict_df(ctx, prediction_length=7, quantile_levels=QS, id_column='id',
                                      timestamp_column='timestamp', target='target', batch_size=256)
                p = p[p.timestamp == day].rename(columns={'id': 'series_id', 'timestamp': 'dt'})
                p = p.rename(columns={str(q): f'q{int(round(q * 100)):02d}' for q in QS})
                out.append(p[['series_id', 'dt'] + [f'q{int(round(q * 100)):02d}' for q in QS]]
                           .assign(model='chronos2', version=ver, wk=w))
    f = pd.concat(out, ignore_index=True)
    qcols = [c for c in f.columns if c.startswith('q')]
    f[qcols] = np.sort(f[qcols].clip(lower=0).values, axis=1)   # keep quantiles in order, no negatives
    keep = dd[['series_id', 'dt', 'sales', 'demand', 'oos_hours']]
    return f.merge(keep, on=['series_id', 'dt'], how='left')

fq = cached('chronos2_quantiles_backtest', make_quantile_forecasts)
fq.to_parquet(PROC_DIR / 'forecasts_stream4_quantiles_v1.parquet', index=False)

# Check: share of real sales below each quantile on in-stock days (should be close to the level)
ins = fq[fq.oos_hours == 0]
print(fq.shape)
print(pd.DataFrame({v: {q: round((g.sales <= g[q]).mean(), 3) for q in ['q10', 'q30', 'q50', 'q90']}
                    for v, g in ins.groupby('version')}))

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Saved chronos2_quantiles_backtest to Drive
(140000, 19)
     corrected    raw
q10      0.150  0.088
q30      0.418  0.300
q50      0.636  0.515
q90      0.950  0.915


## Step 18: Chronos-2 on the second retailer (Rohlik)
Stream 3 built censored sales for Rohlik (adjusted sales × availability) and forecast them with seasonal naive and LightGBM. We add zero-shot Chronos-2 with the same products, weeks, forecast origins (the Sunday before each week) and truth (Rohlik's adjusted sales), to test whether the foundation-model result holds on a second retailer.

In [7]:
!pip install -q "chronos-forecasting>=2.0"

In [8]:
from chronos import Chronos2Pipeline
r = pd.read_parquet(PROC_DIR / 'stream3_rohlik_daily_v1.parquet',
                    columns=['unique_id', 'date', 'adjusted_sales', 'synthetic_raw_sales', 'availability', 'set'])
r['date'] = pd.to_datetime(r['date'])
fc3 = pd.read_parquet(PROC_DIR / 'forecasts_stream3_v1.parquet')
fc3['date'] = pd.to_datetime(fc3['date'])
r_weeks = {'Valid 1': '2024-05-13', 'Valid 2': '2024-05-20', 'Test': '2024-05-27'}
pipe_r = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

def daily_grid(ctx, origin):
    """Every series on a full daily calendar up to the origin; gaps (e.g. closed days) filled by interpolation."""
    full = (ctx.groupby('id').timestamp.min().rename('start').reset_index()
               .assign(end=origin))
    idx = pd.concat([pd.DataFrame({'id': row.id, 'timestamp': pd.date_range(row.start, row.end, freq='D')})
                     for row in full.itertuples()], ignore_index=True)
    g = idx.merge(ctx, on=['id', 'timestamp'], how='left')
    g['target'] = g.groupby('id').target.transform(lambda s: s.interpolate(limit_direction='both'))
    return g

def make_rohlik_forecasts():
    out = []
    for w, a in r_weeks.items():
        origin = pd.Timestamp(a) - pd.Timedelta(days=1)          # the Sunday before the week
        for col, ver in [('synthetic_raw_sales', 'raw'), ('adjusted_sales', 'corrected')]:
            ctx = r[r.date <= origin][['unique_id', 'date', col]].rename(
                columns={'unique_id': 'id', 'date': 'timestamp', col: 'target'})
            p = pipe_r.predict_df(daily_grid(ctx, origin), prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],
                                  id_column='id', timestamp_column='timestamp', target='target', batch_size=256)
            p = p.rename(columns={'id': 'unique_id', 'timestamp': 'date', 'predictions': 'forecast'})
            out.append(p[['unique_id', 'date', 'forecast']].assign(model='chronos2', version=ver))
    f = pd.concat(out, ignore_index=True); f['forecast'] = f.forecast.clip(lower=0)
    return f

fr = cached('chronos2_rohlik', make_rohlik_forecasts)

# Score all three models the same way: against adjusted sales, on the days Stream 3 scored
truth = r[['unique_id', 'date', 'adjusted_sales', 'availability']]
key = fc3[['unique_id', 'date']].drop_duplicates()
allr = pd.concat([fc3[['unique_id', 'date', 'model', 'version', 'forecast']],
                  fr.merge(key, on=['unique_id', 'date'])], ignore_index=True).merge(truth, on=['unique_id', 'date'])
allr['wk'] = None
for w, a in r_weeks.items():
    a = pd.Timestamp(a); allr.loc[allr.date.between(a, a + pd.Timedelta(days=6)), 'wk'] = w
allr['stockout_day'] = allr.availability < 1

rows = []
for (m, v, w), g in allr.groupby(['model', 'version', 'wk']):
    so = g[g.stockout_day]
    rows.append({'model': m, 'version': v, 'wk': w,
                 'WAPE': (g.adjusted_sales - g.forecast).abs().sum() / g.adjusted_sales.sum(),
                 'bias': (g.forecast - g.adjusted_sales).sum() / g.adjusted_sales.sum(),
                 'WAPE stockout days': (so.adjusted_sales - so.forecast).abs().sum() / so.adjusted_sales.sum()})
rr = pd.DataFrame(rows)
print('Test week:')
display(rr[rr.wk == 'Test'].drop(columns='wk').round(3).sort_values('WAPE'))
print('Average of 3 weeks:')
display(rr.groupby(['model', 'version']).mean(numeric_only=True).round(3).sort_values('WAPE'))
print('Weeks (of 3) where corrected beats raw:')
for m, gm in rr.groupby('model'):
    p = gm.pivot(index='wk', columns='version', values='WAPE')
    print(' ', m, int((p.corrected < p.raw).sum()))

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Saved chronos2_rohlik to Drive
Test week:


,model,version,WAPE,bias,WAPE stockout days
0,chronos2,corrected,0.222,-0.059,0.230
3,chronos2,raw,0.232,-0.091,0.288
12,seasonal_naive,corrected,0.272,-0.015,0.264
6,lightgbm,corrected,0.281,-0.044,0.257
15,seasonal_naive,raw,0.282,-0.050,0.316
9,lightgbm,raw,0.288,-0.084,0.291


Average of 3 weeks:


WAPE   bias  WAPE stockout days
model          version                                    
chronos2       corrected  0.203 -0.053               0.208
               raw        0.213 -0.084               0.257
lightgbm       corrected  0.274 -0.026               0.231
seasonal_naive corrected  0.278  0.016               0.278
lightgbm       raw        0.279 -0.062               0.260
seasonal_naive raw        0.285 -0.024               0.313

Weeks (of 3) where corrected beats raw:
  chronos2 3
  lightgbm 3
  seasonal_naive 3
